In [ ]:
!pip install -q \
albumentations \
opencv-python-headless \
timm \
grad-cam \
scikit-learn \
seaborn \
torchxrayvision


In [ ]:
import os
import cv2
import json
import math
import torch
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm
import albumentations as A
import torch.optim as optim
import torchxrayvision as xrv
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score, f1_score,
    roc_curve, confusion_matrix,
    roc_auc_score,
    average_precision_score,
)
from google.colab import drive
import matplotlib.pyplot as plt
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader


In [ ]:
drive.mount('/content/drive', force_remount=True)


In [ ]:
#change

CONFIG = {
    "dataset_root": "/content/rsna-pneumonia-dataset-npy",
    "model_name": "DenseNet121",
    "save_dir": "/content/drive/MyDrive/AI_Research/Model_Results/SaliencyRun", # CHANGE
}

os.makedirs(CONFIG["save_dir"], exist_ok=True)

print(f"Directory made: {CONFIG['save_dir']}")


In [ ]:
ZIP_PATH = "/content/drive/MyDrive/AI_Research/rsna-pneumonia-dataset-npy.zip"
EXTRACT_DIR = "/content/rsna-pneumonia-dataset-npy"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall("/content")

print("Dataset extracted")
DRIVE_ROOT = "/content/drive/MyDrive/AI_Research"

for fname in [
    "test_split.csv",
    "train_split.csv",
    "val_split.csv",
    "stage_2_train_labels.csv"
]:
    src = os.path.join(DRIVE_ROOT, fname)
    dst = os.path.join(EXTRACT_DIR, fname)

    shutil.copy2(src, dst)
    print(f"Copied {fname}")


In [ ]:
SAVE_DIR = CONFIG["save_dir"]

os.makedirs(SAVE_DIR, exist_ok=True)

os.makedirs(
    os.path.join(SAVE_DIR, "gradcam"),
    exist_ok=True
)

os.makedirs(
    os.path.join(SAVE_DIR, "misclassified"),
    exist_ok=True
)

print(SAVE_DIR)


In [ ]:
train_ds = pd.read_csv(
    "/content/rsna-pneumonia-dataset-npy/train_split.csv"
)

val_ds = pd.read_csv(
    "/content/rsna-pneumonia-dataset-npy/val_split.csv"
)

test_ds = pd.read_csv(
    "/content/rsna-pneumonia-dataset-npy/test_split.csv"
)

print(len(train_ds))
print(len(val_ds))
print(len(test_ds))


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

#change

NPY_MAP_PATH = "/content/drive/MyDrive/AI_Research/npy_map.json"
IMAGES_DIR = "/content/rsna-pneumonia-dataset-npy/images"

if os.path.exists(NPY_MAP_PATH):
    print("Loading existing NPY map.")

    with open(NPY_MAP_PATH, "r") as f:
        npy_map = json.load(f)

else:
    print("Building NPY map.")

    npy_map = {
        f.stem: str(f)
        for f in Path(IMAGES_DIR).rglob("*.npy")
    }

    os.makedirs("/content/drive/MyDrive/AI_Research", exist_ok=True)

    with open(NPY_MAP_PATH, "w") as f:
        json.dump(npy_map, f)

print(f"Loaded {len(npy_map):,} paths")

sample_patient = next(iter(npy_map))
print(sample_patient)
print(npy_map[sample_patient])


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import cv2

def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    cv2.setNumThreads(0)
    cv2.ocl.setUseOpenCL(False)
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


In [ ]:
def collate_fn(batch):

    images = torch.stack(
        [x["image"] for x in batch]
    )
    labels = torch.stack(
        [x["label"] for x in batch]
    )
    bboxes = torch.stack(
        [x["bbox"] for x in batch]
    )
    patient_ids = [
        x["patient_id"]
        for x in batch
    ]
    return {
        "images": images,
        "labels": labels,
        "bboxes": bboxes,
        "patient_ids": patient_ids
    }


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)


In [ ]:
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

def worker_init_fn(worker_id):
    import cv2
    cv2.setNumThreads(0)
    cv2.ocl.setUseOpenCL(False)


In [ ]:
HPARAMS2 = {
    "run_name": "frozen_xrv",
    "img_size": 224,
    "epochs": 25,
    "batch_size": 128,
    "initial_lr": 0.0004448862857672661,
    "weight_decay": 1.3560236266648827e-05,
    "dropout": 0.3366734543720239,
    "cosine_tmax": 25,
    "seeds": [42, 739, 2026, 7, 13, 21, 99, 123, 256, 314, 500, 777, 1000, 1234, 2024, 2025, 3407, 4096, 5555, 8080, 9999],
    "xrv_weights": "densenet121-res224-mimic_nb",
}

XRV_SAVE_DIR = os.path.join(SAVE_DIR, "xrv_pipeline_nb") #change to ch vs nb
os.makedirs(XRV_SAVE_DIR, exist_ok=True)
print(XRV_SAVE_DIR)


In [ ]:
train_transform_xrv = A.Compose([
    A.Rotate(limit=7, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=0,
                       border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.5),
    A.Resize(HPARAMS2["img_size"], HPARAMS2["img_size"]),
])

val_transform_xrv = A.Compose([
    A.Resize(HPARAMS2["img_size"], HPARAMS2["img_size"]),
])


In [ ]:
class RSNADatasetXRV(Dataset):

    def __init__(self, dataframe, npy_map, transform=None):
        self.df = dataframe
        self.npy_map = npy_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patient_id = row["patientId"]

        if patient_id not in self.npy_map:
            raise KeyError(f"Missing NPY for {patient_id}")

        img = np.load(self.npy_map[patient_id]).astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)   # -> [0, 1]
        img = (img * 255.0).clip(0, 255).astype(np.uint8)          # -> uint8 [0, 255], required by CLAHE

        if self.transform:
            img = self.transform(image=img)["image"]               # still uint8 HxW here

        img = img.astype(np.float32)
        img = xrv.datasets.normalize(img, 255)                      # -> XRV's expected range
        img = torch.from_numpy(img).unsqueeze(0).float()            # -> [1, H, W]

        label = torch.tensor(int(row["Target"]), dtype=torch.long)
        bbox = torch.tensor(
            [row["x"], row["y"], row["width"], row["height"]],
            dtype=torch.float32,
        )

        return {
            "image": img,
            "label": label,
            "bbox": bbox,
            "patient_id": patient_id,
        }


In [ ]:
train_dataset_xrv = RSNADatasetXRV(train_ds, npy_map, transform=train_transform_xrv)
val_dataset_xrv = RSNADatasetXRV(val_ds, npy_map, transform=val_transform_xrv)
test_dataset_xrv = RSNADatasetXRV(test_ds, npy_map, transform=val_transform_xrv)


In [ ]:
sample = train_dataset_xrv[0]
print("image shape (expect [1, 224, 224]):", sample["image"].shape)
print("image dtype:", sample["image"].dtype)
print("image value range (expect roughly -1024 to 1024):",
      sample["image"].min().item(), sample["image"].max().item())
print("label:", sample["label"].item())


In [ ]:
class XRVClassifier(nn.Module):
    def __init__(self, xrv_weights, dropout, num_classes=2):
        super().__init__()
        backbone = xrv.models.DenseNet(weights=xrv_weights)
        self.features = backbone.features
        num_features = backbone.classifier.in_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        feats = self.features(x)
        feats = torch.relu(feats)
        feats = feats.mean(dim=(2,3))
        return self.classifier(feats)

# sanity check
_sanity_model = XRVClassifier(HPARAMS2["xrv_weights"], HPARAMS2["dropout"]).to(device)
_dummy = torch.zeros(2, 1, HPARAMS2["img_size"], HPARAMS2["img_size"]).to(device)
with torch.no_grad():
    _out = _sanity_model(_dummy)
print("sanity output shape (must be [2, 2]):", _out.shape)
del _sanity_model, _dummy, _out
torch.cuda.empty_cache()


In [ ]:
labels_arr = train_ds["Target"].values.astype(np.int64)
counts = np.bincount(labels_arr)
neg, pos = counts[0], counts[1]
pos_weight_ratio = neg / (pos + 1e-8)
class_weights = torch.tensor([1.0, pos_weight_ratio], dtype=torch.float32).to(device)

criterion_xrv = nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
@torch.no_grad()
def evaluate_xrv(model, loader, threshold=0.5):
    model.eval()
    losses, y_true, y_pred, y_prob = [], [], [], []

    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, dtype=torch.long)

        outputs = model(images)
        loss = criterion_xrv(outputs, labels)
        losses.append(loss.item())

        probs = torch.softmax(outputs.float(), dim=1)[:, 1]
        preds = (probs >= threshold).int()

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)

    return {
        "loss": np.mean(losses),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "y_true": y_true,
        "y_prob": y_prob,
    }


In [ ]:
class GradCAM(nn.Module):
    def __init__(self, model):
        super(GradCAM, self).__init__()
        self.densenet = model
        self.features_conv = model.features
        self.pool = model.pool
        self.classifier = model.classifier
        self.gradients = None

    def activations_hook(self, grad):
        self.gradients = grad

    def forward(self, x):
        x = self.features_conv(x)
        x = torch.relu(x)
        x.register_hook(self.activations_hook)
        x = x.mean(dim=(2, 3))
        return self.classifier(x)

    def get_activations_gradient(self):
        return self.gradients

    def get_activations(self, x):
        return torch.relu(self.features_conv(x))


In [ ]:
def train_one_run(seed, run_name):

    ckpt_path    = os.path.join(XRV_SAVE_DIR, f"{run_name}_model.pth")     # resume
    best_path    = os.path.join(XRV_SAVE_DIR, f"{run_name}_best.pth")      # peak val AUC
    final_path   = os.path.join(XRV_SAVE_DIR, f"{run_name}_epoch25.pth")   # last epoch
    metrics_path = os.path.join(XRV_SAVE_DIR, f"{run_name}_best_metrics.json")
    hist_path    = os.path.join(XRV_SAVE_DIR, f"{run_name}_history.json")

    if os.path.exists(metrics_path) and os.path.exists(ckpt_path):
        with open(metrics_path) as f:
            prev = json.load(f)
        start_epoch  = prev["last_epoch"]
        best_val_auc = prev.get("best_val_auc", prev.get("val_roc_auc", -1.0))
        best_epoch   = prev.get("best_epoch", prev["last_epoch"])
        resuming = True
        print(f"[{run_name}] Resuming from epoch {start_epoch} "
              f"(best so far {best_val_auc:.4f} @ epoch {best_epoch})")
    else:
        start_epoch = 0
        best_val_auc = -1.0
        best_epoch = 0
        resuming = False
        print(f"[{run_name}] Starting fresh (seed={seed})")

    set_determinism(seed)

    model = XRVClassifier(HPARAMS2["xrv_weights"], HPARAMS2["dropout"]).to(device)
    if resuming:
        model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
        model = model.to(device)

    optimizer = optim.AdamW(
        model.parameters(), lr=HPARAMS2["initial_lr"],
        weight_decay=HPARAMS2["weight_decay"]
    )
    for group in optimizer.param_groups:
        group["initial_lr"] = HPARAMS2["initial_lr"]

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=HPARAMS2["cosine_tmax"],
        last_epoch=start_epoch if resuming else -1
    )
    if resuming:
        correct_lr = HPARAMS2["initial_lr"] * (
            1 + math.cos(math.pi * start_epoch / HPARAMS2["cosine_tmax"])) / 2
        for group in optimizer.param_groups:
            group["lr"] = correct_lr
        print(f"[{run_name}] Forced resume LR: {correct_lr:.3e}")

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset_xrv, batch_size=HPARAMS2["batch_size"], shuffle=True,
        generator=g, collate_fn=collate_fn, num_workers=8, pin_memory=True,
        worker_init_fn=seed_worker, persistent_workers=False
    )

    val_loader = DataLoader(
        val_dataset_xrv, batch_size=HPARAMS2["batch_size"], shuffle=False,
        collate_fn=collate_fn, num_workers=8, pin_memory=True,
        worker_init_fn=seed_worker, persistent_workers=False
    )

    scaler = GradScaler("cuda")

    if resuming and os.path.exists(hist_path):
        with open(hist_path) as f:
            history = json.load(f)
    else:
        history = {k: [] for k in [
            "train_loss", "val_loss", "val_accuracy", "val_precision",
            "val_recall", "val_f1", "val_roc_auc", "val_pr_auc", "learning_rate"
        ]}

    for epoch in range(start_epoch, HPARAMS2["epochs"]):
        current_epoch = epoch + 1
        print(f"\n[{run_name}] Epoch {current_epoch}/{HPARAMS2['epochs']}  "
              f"(LR = {optimizer.param_groups[0]['lr']:.3e})")

        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"{run_name} epoch {current_epoch}", unit="batch")

        for batch in pbar:
            images = batch["images"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda"):
                outputs = model(images)
                loss = criterion_xrv(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader)
        val_metrics = evaluate_xrv(model, val_loader)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_f1"].append(val_metrics["f1"])
        history["val_roc_auc"].append(val_metrics["roc_auc"])
        history["val_pr_auc"].append(val_metrics["pr_auc"])
        history["learning_rate"].append(optimizer.param_groups[0]["lr"])

        print(
            f"Train Loss   : {train_loss:.4f}  (val gap: {val_metrics['loss'] - train_loss:+.4f})\n"
            f"Val ROC-AUC  : {val_metrics['roc_auc']:.4f}\n"
            f"Val PR-AUC   : {val_metrics['pr_auc']:.4f}\n"
            f"Val Recall   : {val_metrics['recall']:.4f}"
        )

        with open(hist_path, "w") as f:
            json.dump(history, f, indent=4)

        torch.save(model.state_dict(), ckpt_path)

        # best-epoch checkpoint: peak validation ROC-AUC
        if val_metrics["roc_auc"] > best_val_auc:
            best_val_auc = float(val_metrics["roc_auc"])
            best_epoch = current_epoch
            torch.save(model.state_dict(), best_path)
            print(f"[{run_name}] New best -> {os.path.basename(best_path)} "
                  f"(epoch {best_epoch}, AUC {best_val_auc:.4f})")

        if current_epoch == HPARAMS2["epochs"]:
            torch.save(model.state_dict(), final_path)
            print(f"[{run_name}] Final -> {os.path.basename(final_path)} "
                  f"(epoch {current_epoch}, AUC {val_metrics['roc_auc']:.4f})")

        with open(metrics_path, "w") as f:
            json.dump({
                "run_name": run_name,
                "seed": seed,
                "last_epoch": current_epoch,
                "last_val_roc_auc": float(val_metrics["roc_auc"]),
                "last_val_pr_auc": float(val_metrics["pr_auc"]),
                "last_val_recall": float(val_metrics["recall"]),
                "best_epoch": best_epoch,
                "best_val_auc": best_val_auc,
            }, f, indent=4)
        print(f"[{run_name}] Saved epoch {current_epoch} "
              f"(best so far: epoch {best_epoch}, AUC {best_val_auc:.4f})")

    print(f"[{run_name}] Done. Best epoch: {best_epoch}, "
          f"Best Val ROC-AUC: {best_val_auc:.4f}")

    test_loader = DataLoader(
        test_dataset_xrv, batch_size=HPARAMS2["batch_size"], shuffle=False,
        collate_fn=collate_fn, num_workers=8, pin_memory=True,
        worker_init_fn=seed_worker, persistent_workers=False
    )

    test_results = {"seed": seed, "run_name": run_name,
                    "best_epoch": best_epoch, "best_val_auc": best_val_auc}

    for tag, path in [("best", best_path), ("epoch25", final_path)]:
        if not os.path.exists(path):
            continue
        model.load_state_dict(torch.load(path, map_location="cpu"))
        model = model.to(device).eval()
        tm = evaluate_xrv(model, test_loader)
        test_results[tag] = {
            "test_roc_auc":   float(tm["roc_auc"]),
            "test_pr_auc":    float(tm["pr_auc"]),
            "test_accuracy":  float(tm["accuracy"]),
            "test_precision": float(tm["precision"]),
            "test_recall":    float(tm["recall"]),
            "test_f1":        float(tm["f1"]),
        }
        np.savez_compressed(
            os.path.join(XRV_SAVE_DIR, f"{run_name}_{tag}_test_probs.npz"),
            y_true=tm["y_true"], y_prob=tm["y_prob"])
        print(f"[{run_name}] {tag:8s} test ROC-AUC {tm['roc_auc']:.4f}")

    test_metrics_path = os.path.join(XRV_SAVE_DIR, f"{run_name}_test_metrics.json")
    with open(test_metrics_path, "w") as f:
        json.dump(test_results, f, indent=4)

    return best_path


In [ ]:
# Rehearsal / bit-identical determinism check

import hashlib

def hash_checkpoint(path):

    sd = torch.load(path, map_location="cpu")
    h = hashlib.sha256()
    for key in sorted(sd.keys()):
        h.update(key.encode())
        h.update(sd[key].contiguous().numpy().tobytes())
    return h.hexdigest()

_orig_epochs = HPARAMS2["epochs"]
_orig_tmax   = HPARAMS2["cosine_tmax"]

REHEARSAL_NAME = "rehearsal_seed42"
rehearsal_ckpt = os.path.join(XRV_SAVE_DIR, f"{REHEARSAL_NAME}_model.pth")

def _clean_rehearsal():
    for suffix in ["_model.pth", "_best.pth", "_epoch25.pth",
                   "_best_metrics.json", "_history.json", "_test_metrics.json",
                   "_best_test_probs.npz", "_epoch25_test_probs.npz"]:
        p = os.path.join(XRV_SAVE_DIR, f"{REHEARSAL_NAME}{suffix}")
        if os.path.exists(p):
            os.remove(p)

try:
    HPARAMS2["epochs"] = 5
    HPARAMS2["cosine_tmax"] = 5          # keep the LR curve self-consistent

    _clean_rehearsal()
    print("=== Rehearsal Run 1 ===")
    train_one_run(seed=42, run_name=REHEARSAL_NAME)
    hash1 = hash_checkpoint(rehearsal_ckpt)
    print(f"Run 1 SHA-256: {hash1}")

    _clean_rehearsal()
    print("\n=== Rehearsal Run 2 ===")
    train_one_run(seed=42, run_name=REHEARSAL_NAME)
    hash2 = hash_checkpoint(rehearsal_ckpt)
    print(f"Run 2 SHA-256: {hash2}")

    assert hash1 == hash2, f"NOT bit-identical!\n  Run1: {hash1}\n  Run2: {hash2}"
    print("\nBit-identical")

finally:
    HPARAMS2["epochs"]      = _orig_epochs
    HPARAMS2["cosine_tmax"] = _orig_tmax
    _clean_rehearsal()
    print(f"restored: epochs={HPARAMS2['epochs']}, "
          f"cosine_tmax={HPARAMS2['cosine_tmax']}")


In [ ]:
import sys

_orig_hook = sys.unraisablehook
def _quiet_unraisable(unraisable):
    if isinstance(unraisable.exc_value, AssertionError) and \
       "child process" in str(unraisable.exc_value):
        return
    _orig_hook(unraisable)
sys.unraisablehook = _quiet_unraisable

ensemble_ckpts = []

for seed in HPARAMS2["seeds"]:
    run_name = f"{HPARAMS2['run_name']}_seed{seed}"
    ckpt = train_one_run(seed, run_name)
    ensemble_ckpts.append(ckpt)

print("\nEnsemble checkpoints:")
for c in ensemble_ckpts:
    print(" ", c)


best epoch test

In [ ]:
CKPT_TAG = "best"          # flip to "epoch25" and re-run for the other checkpoint

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")
box_patients = set(
    labels_df[(labels_df["Target"] == 1) & (labels_df["width"].notna())]["patientId"].tolist()
)
box_test_df = (
    test_ds[test_ds["patientId"].isin(box_patients)]
    .drop_duplicates(subset="patientId")      # one row per patient, not per box
    .reset_index(drop=True)
)
print(f"Box-bearing test patients: {len(box_test_df)}")

gradcam_root = os.path.join(XRV_SAVE_DIR, f"gradcam_npy_{CKPT_TAG}")
os.makedirs(gradcam_root, exist_ok=True)
print(f"Root: {gradcam_root}")

for seed in HPARAMS2["seeds"]:
    run_name = f"{HPARAMS2['run_name']}_seed{seed}"
    ckpt_path = os.path.join(XRV_SAVE_DIR, f"{run_name}_{CKPT_TAG}.pth")

    if not os.path.exists(ckpt_path):
        print(f"[seed {seed}] {CKPT_TAG} checkpoint missing, skipping")
        continue

    seed_out = os.path.join(gradcam_root, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)

    model_gc = XRVClassifier(HPARAMS2["xrv_weights"], HPARAMS2["dropout"]).to(device)
    state_dict_gc = torch.load(ckpt_path, map_location="cpu")
    missing_keys, unexpected_keys = model_gc.load_state_dict(state_dict_gc, strict=True)
    print(f"[seed {seed}] strict=True load ({CKPT_TAG})")
    print(f"  missing keys    : {missing_keys}")
    print(f"  unexpected keys : {unexpected_keys}")
    model_gc.eval()

    gradcam_model_gc = GradCAM(model_gc).to(device)
    gradcam_model_gc.eval()

    preds_rows = []

    for _, row in tqdm(box_test_df.iterrows(), total=len(box_test_df),
                       desc=f"GradCAM {CKPT_TAG} seed {seed}"):
        patient_id = row["patientId"]
        if patient_id not in npy_map:
            continue

        img_np = np.load(npy_map[patient_id]).astype(np.float32)
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        img_np = val_transform_xrv(image=img_np)["image"]
        img_np = img_np.astype(np.float32)
        img_np = xrv.datasets.normalize(img_np, 255)
        img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).float().to(device)

        logits = gradcam_model_gc(img_tensor)
        prob = torch.softmax(logits.float(), dim=1)[0, 1].item()
        pred = int(prob >= 0.5)

        gradcam_model_gc.zero_grad()
        logits[0, 1].backward()

        gradients = gradcam_model_gc.get_activations_gradient()            # [1, C, 7, 7]
        pooled_g  = gradients.mean(dim=[0, 2, 3])                          # [C]
        activs    = gradcam_model_gc.get_activations(img_tensor).detach()  # [1, C, 7, 7]
        weighted  = activs * pooled_g.view(1, -1, 1, 1)
        raw_cam   = weighted.mean(dim=1).squeeze().cpu().numpy()           # raw 7x7, no ReLU

        np.save(os.path.join(seed_out, f"{patient_id}.npy"), raw_cam)

        preds_rows.append({
            "seed": seed,
            "patientId": patient_id,
            "true_label": int(row["Target"]),
            "pred_label": pred,
            "pred_prob": round(prob, 6),
        })

    pd.DataFrame(preds_rows).to_csv(os.path.join(seed_out, "preds.csv"), index=False)
    print(f"[seed {seed}] {len(preds_rows)} CAMs -> {seed_out}")

    del model_gc, gradcam_model_gc
    torch.cuda.empty_cache()

print(f"Grad-CAM export complete ({CKPT_TAG}).")


In [ ]:
import os, json, itertools
import numpy as np
import pandas as pd
import cv2
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

CKPT_TAG   = "best"                     # set "epoch25" to re-run the confirmatory arm
CAM_ROOT   = os.path.join(XRV_SAVE_DIR, f"gradcam_npy_{CKPT_TAG}")
OUT_DIR    = os.path.join(XRV_SAVE_DIR, f"gradcam_metrics_{CKPT_TAG}")
AGG_DIR    = os.path.join(OUT_DIR, "_aggregate")
os.makedirs(AGG_DIR, exist_ok=True)

SCORE_SIZE = 1024
CAM_NATIVE = 7           # raw map resolution; agreement is computed here
N_BOOT     = 1000
BOOT_SEED  = 42

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")

#helpers
def load_raw(seed, pid):

    f = os.path.join(CAM_ROOT, f"seed_{seed}", f"{pid}.npy")
    if not os.path.exists(f):
        return None
    cam = np.load(f).astype(np.float32)
    return cam / max(cam.max(), 1e-8)

def upsample(cam):
    return cv2.resize(cam, (SCORE_SIZE, SCORE_SIZE), interpolation=cv2.INTER_LINEAR)

def box_mask(pid):

    r = labels_df[(labels_df.patientId == pid) & (labels_df.Target == 1)]
    r = r.dropna(subset=["x", "y", "width", "height"])
    if r.empty:
        return None
    m = np.zeros((SCORE_SIZE, SCORE_SIZE), np.uint8)
    for _, b in r.iterrows():
        x0, y0 = int(b.x), int(b.y)
        x1, y1 = int(b.x + b.width), int(b.y + b.height)
        m[max(y0, 0):min(y1, SCORE_SIZE), max(x0, 0):min(x1, SCORE_SIZE)] = 1
    return m if m.sum() else None

def auprc(mask_flat, map_flat):

    order = np.argsort(-map_flat, kind="stable")
    y = mask_flat[order]
    tp = np.cumsum(y)
    n_pos = tp[-1]
    if n_pos == 0:
        return np.nan
    prec = tp / np.arange(1, y.size + 1, dtype=np.float64)
    return float((prec * y).sum() / n_pos)

# whats on the disk
pred_paths = {s: os.path.join(CAM_ROOT, f"seed_{s}", "preds.csv")
              for s in HPARAMS2["seeds"]}
pred_paths = {s: p for s, p in pred_paths.items() if os.path.exists(p)}
SEEDS = sorted(pred_paths.keys())

preds = pd.concat([pd.read_csv(p) for p in pred_paths.values()])
patients = sorted(preds.patientId.unique())
masks = {pid: box_mask(pid) for pid in patients}
masks = {k: v for k, v in masks.items() if v is not None}

print(f"[{CKPT_TAG}] {len(SEEDS)} seeds, {len(patients)} patients, "
      f"{len(masks)} with usable boxes")
if len(SEEDS) < len(HPARAMS2["seeds"]):
    print(f"  [!] missing: {sorted(set(HPARAMS2['seeds']) - set(SEEDS))}")

# score every (seed, image)
rows, zero_maps = [], []

for i, (pid, gt) in enumerate(masks.items(), 1):
    gt_flat = gt.reshape(-1)
    prevalence = float(gt.mean())          # exact AUPRC of a random ranker
    for s in SEEDS:
        raw = load_raw(s, pid)
        if raw is None:
            continue
        if raw.max() <= 0:
            zero_maps.append({"seed": s, "patientId": pid})
            rows.append({"seed": s, "patientId": pid, "auprc": 0.0,
                         "pointing_hit": 0, "prevalence": prevalence,
                         "all_zero": True})
            continue
        m = upsample(raw)
        py, px = np.unravel_index(np.argmax(m), m.shape)
        rows.append({"seed": s, "patientId": pid,
                     "auprc": auprc(gt_flat, m.reshape(-1)),
                     "pointing_hit": int(gt[py, px] == 1),
                     "prevalence": prevalence,
                     "all_zero": False})
    if i % 50 == 0:
        print(f"  {i}/{len(masks)} images scored")

sdf = pd.DataFrame(rows)
sdf.to_csv(os.path.join(AGG_DIR, "per_image_per_seed_scores.csv"), index=False)
print(f"\nscored {len(sdf)} (seed, image) pairs; {len(zero_maps)} all-zero maps")

# per seed output
per_seed_rows = []

for seed in SEEDS:
    sub = sdf[sdf.seed == seed]
    preds_s = pd.read_csv(pred_paths[seed])[["patientId", "pred_prob"]]
    merged = sub.merge(preds_s, on="patientId", how="left")

    seed_out = os.path.join(OUT_DIR, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)
    merged.to_csv(os.path.join(seed_out, "localization.csv"), index=False)

    pos = merged[merged.pred_prob >= 0.5]
    s_sum = {
        "seed": seed, "checkpoint": CKPT_TAG, "n_images": len(merged),
        "auprc_mean": float(merged.auprc.mean()),
        "auprc_sd_across_images": float(merged.auprc.std()),
        "auprc_median": float(merged.auprc.median()),
        "pointing_hit": float(merged.pointing_hit.mean()),
        "auprc_pred_positive": float(pos.auprc.mean()) if len(pos) else None,
        "n_pred_positive": int(len(pos)),
        "mean_pred_prob": float(merged.pred_prob.mean()),
    }
    with open(os.path.join(seed_out, "summary.json"), "w") as f:
        json.dump(s_sum, f, indent=4)

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].hist(merged.auprc, bins=30, color="steelblue")
    ax[0].axvline(merged.prevalence.mean(), color="r", ls="--",
                  label=f"random {merged.prevalence.mean():.3f}")
    ax[0].set_title(f"seed {seed} ({CKPT_TAG}): per-image AUPRC")
    ax[0].set_xlabel("AUPRC"); ax[0].legend(fontsize=8)
    ax[1].hist(merged.pred_prob, bins=25, color="slategray")
    ax[1].axvline(0.5, color="r", ls="--")
    ax[1].set_title(f"seed {seed}: p(pneumonia)"); ax[1].set_xlabel("prob")
    plt.tight_layout()
    plt.savefig(os.path.join(seed_out, "metrics.png"), dpi=140, bbox_inches="tight")
    plt.close()

    per_seed_rows.append(s_sum)
    print(f"  seed {seed:>5}: auprc {s_sum['auprc_mean']:.4f}  "
          f"pointing {s_sum['pointing_hit']:.4f}")

psdf = pd.DataFrame(per_seed_rows)
psdf.to_csv(os.path.join(AGG_DIR, "per_seed_summary.csv"), index=False)

# aggregates
per_seed = psdf.set_index("seed").auprc_mean
spread = float(per_seed.std(ddof=1))          # SD of per-seed means

rng = np.random.default_rng(BOOT_SEED)
boot = np.array([pd.Series(rng.choice(per_seed.values, len(per_seed), replace=True))
                 .std(ddof=1) for _ in range(N_BOOT)])
lo, hi = np.percentile(boot, [2.5, 97.5])

per_img = sdf.groupby("patientId").auprc.agg(["mean", "std"])
q1, med, q3 = per_img["std"].quantile([0.25, 0.50, 0.75])

yy, xx = np.mgrid[0:SCORE_SIZE, 0:SCORE_SIZE]
c = SCORE_SIZE / 2
centre_map = np.exp(-((xx - c) ** 2 + (yy - c) ** 2) / (2 * (SCORE_SIZE / 5) ** 2))
centre_flat = centre_map.astype(np.float32).reshape(-1)
centre_scores = [auprc(gt.reshape(-1), centre_flat) for gt in masks.values()]

print(f"[{CKPT_TAG}] Primary: per-image AUPRC (threshold-free)")
print(psdf[["seed", "auprc_mean", "pointing_hit", "auprc_pred_positive"]]
      .round(4).to_string(index=False))
print(f"\n  mean across seeds        : {per_seed.mean():.4f}")
print(f"  Spread (SD of per-seed)  : {spread:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  per-seed range           : {per_seed.min():.4f} - {per_seed.max():.4f}")
print(f"\n  random-ranker baseline   : {sdf.prevalence.mean():.4f} "
      f"(= mean positive-pixel rate)")
print(f"  centred-Gaussian baseline: {np.mean(centre_scores):.4f}")
print(f"\n  per-image SD across seeds: median {med:.4f}  IQR [{q1:.4f}, {q3:.4f}]")
print(f"  pointing hit rate        : {psdf.pointing_hit.mean():.4f} "
      f"+/- {psdf.pointing_hit.std():.4f}")

# inter-seed agreement, computed at 7x7
pairs = list(itertools.combinations(SEEDS, 2))
agr_rows = []
px_per_cell = SCORE_SIZE / CAM_NATIVE

for pid in masks:
    cams = {s: load_raw(s, pid) for s in SEEDS}
    cams = {s: c for s, c in cams.items() if c is not None}
    if len(cams) < 2:
        continue
    for a, b in pairs:
        if a not in cams or b not in cams:
            continue
        rho, _ = spearmanr(cams[a].ravel(), cams[b].ravel())
        ya, xa = np.unravel_index(np.argmax(cams[a]), cams[a].shape)
        yb, xb = np.unravel_index(np.argmax(cams[b]), cams[b].shape)
        agr_rows.append({"patientId": pid, "seed_a": a, "seed_b": b,
                         "spearman": float(rho),
                         "peak_px": float(np.hypot(ya - yb, xa - xb) * px_per_cell)})

adf = pd.DataFrame(agr_rows)
adf.to_csv(os.path.join(AGG_DIR, "cam_agreement.csv"), index=False)

wide = preds.pivot_table(index="patientId", columns="seed", values="pred_label")
unanimous = float((wide.nunique(axis=1) == 1).mean())

print(f"[{CKPT_TAG}] inter-seed agreement ({len(pairs)} pairs, at 7x7 native)")
print(f"  spearman  {adf.spearman.mean():.4f} +/- {adf.spearman.std():.4f}")
print(f"  peak_px   {adf.peak_px.mean():.1f} +/- {adf.peak_px.std():.1f} "
      f"(in 1024-px units)")
print(f"  seeds unanimous on the label (thr 0.5): {unanimous:.1%}")
print(f"  mean prob SD across seeds: "
      f"{preds.groupby('patientId').pred_prob.std().mean():.4f}")

#save
summary = {
    "status": "exploratory (best checkpoint; §7 puts this out of scope)",
    "checkpoint": CKPT_TAG, "score_size": SCORE_SIZE,
    "n_seeds": len(SEEDS), "n_images": len(masks),
    "n_all_zero_maps": len(zero_maps),
    "auprc": {"mean_across_seeds": float(per_seed.mean()),
              "spread_sd_of_per_seed_mean": spread,
              "bootstrap_ci_95": [float(lo), float(hi)],
              "per_seed_min": float(per_seed.min()),
              "per_seed_max": float(per_seed.max())},
    "per_image_sd": {"median": float(med), "iqr": [float(q1), float(q3)]},
    "baseline_random": float(sdf.prevalence.mean()),
    "baseline_centre_gaussian": float(np.mean(centre_scores)),
    "pointing_hit": {"mean": float(psdf.pointing_hit.mean()),
                     "std": float(psdf.pointing_hit.std())},
    "agreement": {"spearman": {"mean": float(adf.spearman.mean()),
                               "std": float(adf.spearman.std())},
                  "peak_px": {"mean": float(adf.peak_px.mean()),
                              "std": float(adf.peak_px.std())}},
    "label_unanimity": unanimous,
}
with open(os.path.join(AGG_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].bar(range(len(psdf)), psdf.auprc_mean, color="steelblue")
ax[0].axhline(sdf.prevalence.mean(), color="r", ls="--", label="random baseline")
ax[0].axhline(np.mean(centre_scores), color="orange", ls="--", label="centre baseline")
ax[0].set_xticks(range(len(psdf)))
ax[0].set_xticklabels(psdf.seed, rotation=90, fontsize=7)
ax[0].set_ylabel("mean AUPRC"); ax[0].set_title(f"per-seed AUPRC ({CKPT_TAG})")
ax[0].legend(fontsize=8)

ax[1].hist(per_img["std"], bins=40, color="seagreen")
ax[1].axvline(med, color="r", ls="--", label=f"median {med:.3f}")
ax[1].set_xlabel("per-image SD across seeds")
ax[1].set_title("per-image variance"); ax[1].legend(fontsize=8)

ax[2].bar(range(len(psdf)), psdf.pointing_hit, color="indianred")
ax[2].set_xticks(range(len(psdf)))
ax[2].set_xticklabels(psdf.seed, rotation=90, fontsize=7)
ax[2].set_ylabel("hit rate"); ax[2].set_title(f"pointing game ({CKPT_TAG})")

plt.tight_layout()
plt.savefig(os.path.join(AGG_DIR, "metrics.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nper-seed  -> {OUT_DIR}/seed_*/")
print(f"aggregate -> {AGG_DIR}")


epoch 25 grad

In [ ]:
# Grad-CAM export loop for epoch 25 checkpoints

CKPT_TAG = "epoch25"
FOLDER_25 = "gradcam_npy_25"
CAM_ROOT = os.path.join(XRV_SAVE_DIR, "gradcam_npy_25")

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")
box_patients = set(
    labels_df[(labels_df["Target"] == 1) & (labels_df["width"].notna())]["patientId"].tolist()
)
box_test_df = (
    test_ds[test_ds["patientId"].isin(box_patients)]
    .drop_duplicates(subset="patientId")     # one row per patient, not per box
    .reset_index(drop=True)
)
print(f"Box-bearing test patients: {len(box_test_df)}")

gradcam_root_25 = os.path.join(XRV_SAVE_DIR, FOLDER_25)
os.makedirs(gradcam_root_25, exist_ok=True)
print(f"Root: {gradcam_root_25}")

for seed in HPARAMS2["seeds"]:
    run_name = f"{HPARAMS2['run_name']}_seed{seed}"
    ckpt_path = os.path.join(XRV_SAVE_DIR, f"{run_name}_{CKPT_TAG}.pth")

    if not os.path.exists(ckpt_path):
        print(f"[seed {seed}] {CKPT_TAG} checkpoint missing, skipping")
        continue

    seed_out = os.path.join(gradcam_root_25, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)

    model_gc = XRVClassifier(HPARAMS2["xrv_weights"], HPARAMS2["dropout"]).to(device)
    state_dict_gc = torch.load(ckpt_path, map_location="cpu")
    missing_keys, unexpected_keys = model_gc.load_state_dict(state_dict_gc, strict=True)
    print(f"[seed {seed}] strict=True load ({CKPT_TAG})")
    print(f"  missing keys    : {missing_keys}")
    print(f"  unexpected keys : {unexpected_keys}")
    model_gc.eval()

    gradcam_model_gc = GradCAM(model_gc).to(device)
    gradcam_model_gc.eval()

    preds_rows = []

    for _, row in tqdm(box_test_df.iterrows(), total=len(box_test_df),
                       desc=f"GradCAM ep25 seed {seed}"):
        patient_id = row["patientId"]
        if patient_id not in npy_map:
            continue

        img_np = np.load(npy_map[patient_id]).astype(np.float32)
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        img_np = val_transform_xrv(image=img_np)["image"]
        img_np = img_np.astype(np.float32)
        img_np = xrv.datasets.normalize(img_np, 255)
        img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).float().to(device)

        logits = gradcam_model_gc(img_tensor)
        prob = torch.softmax(logits.float(), dim=1)[0, 1].item()
        pred = int(prob >= 0.5)

        gradcam_model_gc.zero_grad()
        logits[0, 1].backward()

        gradients = gradcam_model_gc.get_activations_gradient()            # [1, C, 7, 7]
        pooled_g  = gradients.mean(dim=[0, 2, 3])                          # [C]
        activs    = gradcam_model_gc.get_activations(img_tensor).detach()  # [1, C, 7, 7]
        weighted  = activs * pooled_g.view(1, -1, 1, 1)
        raw_cam   = weighted.mean(dim=1).squeeze().cpu().numpy()           # raw 7x7, no ReLU

        np.save(os.path.join(seed_out, f"{patient_id}.npy"), raw_cam)

        preds_rows.append({
            "seed": seed,
            "patientId": patient_id,
            "true_label": int(row["Target"]),
            "pred_label": pred,
            "pred_prob": round(prob, 6),
        })

    pd.DataFrame(preds_rows).to_csv(os.path.join(seed_out, "preds.csv"), index=False)
    print(f"[seed {seed}] {len(preds_rows)} CAMs -> {seed_out}")

    del model_gc, gradcam_model_gc
    torch.cuda.empty_cache()

print(f"Grad-CAM export complete ({CKPT_TAG}).")


In [ ]:
import os, json, itertools
import numpy as np
import pandas as pd
import cv2
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

CKPT_TAG   = "epoch25"
CAM_ROOT   = os.path.join(XRV_SAVE_DIR, "gradcam_npy_25")
MET_ROOT   = os.path.join(XRV_SAVE_DIR, "gradcam_metrics_25")
AGG_DIR    = os.path.join(MET_ROOT, "_aggregate")
os.makedirs(AGG_DIR, exist_ok=True)

SCORE_SIZE = 1024        # §5
CAM_NATIVE = 7           # raw map resolution
N_BOOT     = 1000        # §6.2 / §6.3
BOOT_SEED  = 42

ARUN = {"XRAI": 0.52, "GCAM": 0.41, "GRAD": 0.34,
        "IG": 0.32, "GGCAM": 0.32, "SG": 0.17, "SIG": 0.16}
ARUN_ORDER = ["XRAI", "GCAM", "GRAD", "IG", "GGCAM", "SG", "SIG"]

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")

# helpers
def load_raw(seed, pid):

    f = os.path.join(CAM_ROOT, f"seed_{seed}", f"{pid}.npy")
    if not os.path.exists(f):
        return None
    cam = np.load(f).astype(np.float32)
    return cam / max(cam.max(), 1e-8)

def upsample(cam):
    return cv2.resize(cam, (SCORE_SIZE, SCORE_SIZE), interpolation=cv2.INTER_LINEAR)

def box_mask(pid):

    r = labels_df[(labels_df.patientId == pid) & (labels_df.Target == 1)]
    r = r.dropna(subset=["x", "y", "width", "height"])
    if r.empty:
        return None
    m = np.zeros((SCORE_SIZE, SCORE_SIZE), np.uint8)
    for _, b in r.iterrows():
        x0, y0 = int(b.x), int(b.y)
        x1, y1 = int(b.x + b.width), int(b.y + b.height)
        m[max(y0, 0):min(y1, SCORE_SIZE), max(x0, 0):min(x1, SCORE_SIZE)] = 1
    return m if m.sum() else None

def auprc(mask_flat, map_flat):

    order = np.argsort(-map_flat, kind="stable")
    y = mask_flat[order]
    tp = np.cumsum(y)
    n_pos = tp[-1]
    if n_pos == 0:
        return np.nan
    prec = tp / np.arange(1, y.size + 1, dtype=np.float64)
    return float((prec * y).sum() / n_pos)

# whats on the disk
pred_paths = {s: os.path.join(CAM_ROOT, f"seed_{s}", "preds.csv")
              for s in HPARAMS2["seeds"]}
pred_paths = {s: p for s, p in pred_paths.items() if os.path.exists(p)}
SEEDS = sorted(pred_paths.keys())

preds = pd.concat([pd.read_csv(p) for p in pred_paths.values()])
patients = sorted(preds.patientId.unique())
masks = {pid: box_mask(pid) for pid in patients}
masks = {k: v for k, v in masks.items() if v is not None}

print(f"[{CKPT_TAG}] {len(SEEDS)} seeds, {len(patients)} patients, "
      f"{len(masks)} with usable boxes")
missing_seeds = sorted(set(HPARAMS2["seeds"]) - set(SEEDS))
if missing_seeds:
    print(f"  [!] missing seeds {missing_seeds} -- §6.5 requires documenting these")

# score every (seed, image)
rows, zero_maps = [], []

for i, (pid, gt) in enumerate(masks.items(), 1):
    gt_flat = gt.reshape(-1)
    prevalence = float(gt.mean())          # exact AUPRC of a random ranker
    for s in SEEDS:
        raw = load_raw(s, pid)
        if raw is None:
            continue
        if raw.max() <= 0:                 # §6.5
            zero_maps.append({"seed": s, "patientId": pid})
            rows.append({"seed": s, "patientId": pid, "auprc": 0.0,
                         "pointing_hit": 0, "prevalence": prevalence,
                         "all_zero": True})
            continue
        m = upsample(raw)
        py, px = np.unravel_index(np.argmax(m), m.shape)
        rows.append({"seed": s, "patientId": pid,
                     "auprc": auprc(gt_flat, m.reshape(-1)),
                     "pointing_hit": int(gt[py, px] == 1),
                     "prevalence": prevalence,
                     "all_zero": False})
    if i % 50 == 0:
        print(f"  {i}/{len(masks)} images scored")

sdf = pd.DataFrame(rows)
sdf.to_csv(os.path.join(AGG_DIR, "per_image_per_seed_scores.csv"), index=False)
print(f"\nscored {len(sdf)} (seed, image) pairs; {len(zero_maps)} all-zero maps")

# per seed output
per_seed_rows = []

for seed in SEEDS:
    sub = sdf[sdf.seed == seed]
    preds_s = pd.read_csv(pred_paths[seed])[["patientId", "pred_prob"]]
    merged = sub.merge(preds_s, on="patientId", how="left")

    seed_out = os.path.join(MET_ROOT, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)
    merged.to_csv(os.path.join(seed_out, "localization.csv"), index=False)

    pos = merged[merged.pred_prob >= 0.5]
    s_sum = {
        "seed": seed, "checkpoint": CKPT_TAG, "n_images": len(merged),
        "auprc_mean": float(merged.auprc.mean()),
        "auprc_sd_across_images": float(merged.auprc.std()),
        "auprc_median": float(merged.auprc.median()),
        "pointing_hit": float(merged.pointing_hit.mean()),
        "auprc_pred_positive": float(pos.auprc.mean()) if len(pos) else None,
        "n_pred_positive": int(len(pos)),
        "mean_pred_prob": float(merged.pred_prob.mean()),
    }
    with open(os.path.join(seed_out, "summary.json"), "w") as f:
        json.dump(s_sum, f, indent=4)

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].hist(merged.auprc, bins=30, color="indianred")
    ax[0].axvline(merged.prevalence.mean(), color="r", ls="--",
                  label=f"random {merged.prevalence.mean():.3f}")
    ax[0].set_title(f"seed {seed} (ep25): per-image AUPRC")
    ax[0].set_xlabel("AUPRC"); ax[0].legend(fontsize=8)
    ax[1].hist(merged.pred_prob, bins=25, color="slategray")
    ax[1].axvline(0.5, color="r", ls="--")
    ax[1].set_title(f"seed {seed} (ep25): p(pneumonia)"); ax[1].set_xlabel("prob")
    plt.tight_layout()
    plt.savefig(os.path.join(seed_out, "metrics.png"), dpi=140, bbox_inches="tight")
    plt.close()

    per_seed_rows.append(s_sum)
    print(f"  seed {seed:>5}: auprc {s_sum['auprc_mean']:.4f}  "
          f"pointing {s_sum['pointing_hit']:.4f}")

psdf = pd.DataFrame(per_seed_rows)
psdf.to_csv(os.path.join(AGG_DIR, "per_seed_summary.csv"), index=False)

# per-image variance
per_img = sdf.groupby("patientId").auprc.agg(["mean", "std"])
q1, med, q3 = per_img["std"].quantile([0.25, 0.50, 0.75])
per_img.to_csv(os.path.join(AGG_DIR, "per_image_variance.csv"))

print("6.1  per-image variance  (SD of AUPRC across seeds, per image)")
print(f"  median SD : {med:.4f}")
print(f"  IQR       : [{q1:.4f}, {q3:.4f}]")
print(f"  min / max : {per_img['std'].min():.4f} / {per_img['std'].max():.4f}")

# C1: spread vs gaps
per_seed = psdf.set_index("seed").auprc_mean
spread = float(per_seed.std(ddof=1))

rng = np.random.default_rng(BOOT_SEED)
boot = np.array([pd.Series(rng.choice(per_seed.values, len(per_seed), replace=True))
                 .std(ddof=1) for _ in range(N_BOOT)])
lo, hi = np.percentile(boot, [2.5, 97.5])

gaps = {f"{ARUN_ORDER[i]}->{ARUN_ORDER[i+1]}":
        round(ARUN[ARUN_ORDER[i]] - ARUN[ARUN_ORDER[i+1]], 4)
        for i in range(len(ARUN_ORDER) - 1)}

yy, xx = np.mgrid[0:SCORE_SIZE, 0:SCORE_SIZE]
c = SCORE_SIZE / 2
centre_map = np.exp(-((xx - c) ** 2 + (yy - c) ** 2) / (2 * (SCORE_SIZE / 5) ** 2))
centre_flat = centre_map.astype(np.float32).reshape(-1)
centre_scores = [auprc(gt.reshape(-1), centre_flat) for gt in masks.values()]

print("C1 test: seed-induced spread vs published adjacent gaps")
print(psdf[["seed", "auprc_mean", "pointing_hit", "auprc_pred_positive"]]
      .round(4).to_string(index=False))
print(f"\n  mean AUPRC across seeds  : {per_seed.mean():.4f}")
print(f"  per-seed range           : {per_seed.min():.4f} - {per_seed.max():.4f}")
print(f"  Spread (SD of per-seed)  : {spread:.4f}")
print(f"  bootstrap 95% CI         : [{lo:.4f}, {hi:.4f}]")
print(f"\n  random-ranker baseline   : {sdf.prevalence.mean():.4f}")
print(f"  centred-Gaussian baseline: {np.mean(centre_scores):.4f}")
print(f"  Arun et al. GCAM         : {ARUN['GCAM']:.2f}")
print("\n  adjacent gap            gap     spread >= gap?")
for name, g in gaps.items():
    print(f"    {name:<22s} {g:.2f}    {'yes' if spread >= g else 'no'}")

n_ge = sum(spread >= g for g in gaps.values())
print(f"\n  spread matches or exceeds {n_ge}/{len(gaps)} adjacent gaps")
print(f"  H1 {'supported' if n_ge > 0 else 'not supported'} "
      f"(unsupported iff spread is small relative to every gap)")

# C2: ranking flips

pairs_rank = [(ARUN_ORDER[i], ARUN_ORDER[i + 1]) for i in range(len(ARUN_ORDER) - 1)]
flip_rows = []

for s in SEEDS:
    scores = dict(ARUN)
    scores["GCAM"] = float(per_seed[s])
    flips = {f"{a}->{b}": int((ARUN[a] > ARUN[b]) != (scores[a] > scores[b]))
             for a, b in pairs_rank}
    new_rank = sorted(ARUN_ORDER, key=lambda m: -scores[m]).index("GCAM") + 1
    flip_rows.append({"seed": s, "gcam_score": scores["GCAM"],
                      "new_rank": new_rank, "rank_shift": new_rank - 2,
                      "n_flips": sum(flips.values()), **flips})

fdf = pd.DataFrame(flip_rows)
fdf.to_csv(os.path.join(AGG_DIR, "c2_flip_matrix.csv"), index=False)

seeds_ge1 = int((fdf.n_flips >= 1).sum())
seeds_ge2 = int((fdf.n_flips >= 2).sum())

print("6.3  C2 test: ranking flips")
print(fdf[["seed", "gcam_score", "new_rank", "rank_shift", "n_flips"]]
      .round(4).to_string(index=False))
print(f"\n  published GCAM rank: 2 of {len(ARUN_ORDER)}")
print(f"  seeds with >=1 adjacent-pair flip : {seeds_ge1}/{len(SEEDS)}")
print(f"  seeds with >=2 adjacent-pair flips: {seeds_ge2}/{len(SEEDS)}")
print(f"  seeds whose GCAM rank changes     : {int((fdf.rank_shift != 0).sum())}"
      f"/{len(SEEDS)}")
print(f"\n  H2 criterion (>=2 pairs in >=3 seeds): "
      f"{'met' if seeds_ge2 >= 3 else 'not met'}")
print("  [see code note: the >=2-pairs criterion may be unreachable by design]")

boot_rank = np.array([
    (pd.Series(rng.choice(fdf.rank_shift.values, len(fdf), replace=True)) != 0).mean()
    for _ in range(N_BOOT)])
rlo, rhi = np.percentile(boot_rank, [2.5, 97.5])
print(f"  rank-change rate {float((fdf.rank_shift != 0).mean()):.3f} "
      f"(bootstrap 95% CI [{rlo:.3f}, {rhi:.3f}])")

# 6.4 model-equivalence control
auc_rows = []
for s in SEEDS:
    rn = f"{HPARAMS2['run_name']}_seed{s}"
    p = os.path.join(XRV_SAVE_DIR, f"{rn}_test_metrics.json")
    if not os.path.exists(p):
        continue
    d = json.load(open(p))
    blk = d.get("epoch25") if isinstance(d.get("epoch25"), dict) else d
    auc_rows.append({"seed": s,
                     "test_roc_auc": blk.get("test_roc_auc"),
                     "test_pr_auc": blk.get("test_pr_auc"),
                     "best_epoch": d.get("best_epoch")})

if auc_rows:
    mdf = pd.DataFrame(auc_rows)
    mdf.to_csv(os.path.join(AGG_DIR, "model_equivalence.csv"), index=False)
    print("6.4  model-equivalence control (epoch-25 checkpoints)")
    print(mdf.round(4).to_string(index=False))
    print(f"\n  test ROC-AUC {mdf.test_roc_auc.mean():.4f} "
          f"+/- {mdf.test_roc_auc.std():.4f}")
    print(f"  test PR-AUC  {mdf.test_pr_auc.mean():.4f} "
          f"+/- {mdf.test_pr_auc.std():.4f}")

# exploratory: inter-seed agreement, computed at 7x7
pairs = list(itertools.combinations(SEEDS, 2))
agr_rows = []
px_per_cell = SCORE_SIZE / CAM_NATIVE

for pid in masks:
    cams = {s: load_raw(s, pid) for s in SEEDS}
    cams = {s: c for s, c in cams.items() if c is not None}
    if len(cams) < 2:
        continue
    for a, b in pairs:
        if a not in cams or b not in cams:
            continue
        rho, _ = spearmanr(cams[a].ravel(), cams[b].ravel())
        ya, xa = np.unravel_index(np.argmax(cams[a]), cams[a].shape)
        yb, xb = np.unravel_index(np.argmax(cams[b]), cams[b].shape)
        agr_rows.append({"patientId": pid, "seed_a": a, "seed_b": b,
                         "spearman": float(rho),
                         "peak_px": float(np.hypot(ya - yb, xa - xb) * px_per_cell)})

adf = pd.DataFrame(agr_rows)
adf.to_csv(os.path.join(AGG_DIR, "cam_agreement.csv"), index=False)

wide = preds.pivot_table(index="patientId", columns="seed", values="pred_label")
unanimous = float((wide.nunique(axis=1) == 1).mean())

print(f"Exploratory: inter-seed agreement ({len(pairs)} pairs, at 7x7 native)")
print(f"  spearman  {adf.spearman.mean():.4f} +/- {adf.spearman.std():.4f}")
print(f"  peak_px   {adf.peak_px.mean():.1f} +/- {adf.peak_px.std():.1f} "
      f"(1024-px units)")
print(f"  seeds unanimous on the label (thr 0.5): {unanimous:.1%}")
print(f"  mean prob SD across seeds: "
      f"{preds.groupby('patientId').pred_prob.std().mean():.4f}")

# save
summary = {
    "status": "confirmatory (epoch-25 final checkpoint, per §5)",
    "plan_version": "v2 2026-08-31",
    "checkpoint": CKPT_TAG, "score_size": SCORE_SIZE,
    "n_seeds": len(SEEDS), "n_images": len(masks),
    "missing_seeds": missing_seeds,
    "n_all_zero_maps": len(zero_maps),
    "primary_auprc": {"mean_across_seeds": float(per_seed.mean()),
                      "spread_sd_of_per_seed_mean": spread,
                      "bootstrap_ci_95": [float(lo), float(hi)],
                      "per_seed_min": float(per_seed.min()),
                      "per_seed_max": float(per_seed.max())},
    "per_image_sd": {"median": float(med), "iqr": [float(q1), float(q3)]},
    "baseline_random": float(sdf.prevalence.mean()),
    "baseline_centre_gaussian": float(np.mean(centre_scores)),
    "arun_gaps": gaps,
    "n_gaps_matched_or_exceeded": n_ge,
    "c2": {"seeds_ge1_flip": seeds_ge1, "seeds_ge2_flips": seeds_ge2,
           "seeds_rank_changed": int((fdf.rank_shift != 0).sum()),
           "criterion_met": bool(seeds_ge2 >= 3)},
    "pointing_hit": {"mean": float(psdf.pointing_hit.mean()),
                     "std": float(psdf.pointing_hit.std())},
    "agreement_exploratory": {
        "spearman": {"mean": float(adf.spearman.mean()),
                     "std": float(adf.spearman.std())},
        "peak_px": {"mean": float(adf.peak_px.mean()),
                    "std": float(adf.peak_px.std())}},
    "label_unanimity": unanimous,
}
with open(os.path.join(AGG_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].bar(range(len(psdf)), psdf.auprc_mean, color="indianred")
ax[0].axhline(ARUN["GCAM"], color="k", ls="--", label="Arun GCAM 0.41")
ax[0].axhline(sdf.prevalence.mean(), color="r", ls=":", label="random baseline")
ax[0].set_xticks(range(len(psdf)))
ax[0].set_xticklabels(psdf.seed, rotation=90, fontsize=7)
ax[0].set_ylabel("mean AUPRC"); ax[0].set_title("per-seed AUPRC (ep25)")
ax[0].legend(fontsize=8)

ax[1].hist(per_img["std"], bins=40, color="seagreen")
ax[1].axvline(med, color="r", ls="--", label=f"median {med:.3f}")
ax[1].set_xlabel("per-image SD across seeds")
ax[1].set_title("6.1 per-image variance"); ax[1].legend(fontsize=8)

ax[2].bar(range(len(gaps)), list(gaps.values()), color="lightgrey",
          label="Arun adjacent gap")
ax[2].axhline(spread, color="crimson", lw=2, label=f"seed spread {spread:.3f}")
ax[2].fill_between([-0.5, len(gaps) - 0.5], lo, hi, color="crimson", alpha=0.15,
                   label="95% CI")
ax[2].set_xticks(range(len(gaps)))
ax[2].set_xticklabels(gaps.keys(), rotation=45, ha="right", fontsize=7)
ax[2].set_title("6.2 C1: spread vs gaps"); ax[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(AGG_DIR, "metrics.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nper-seed  -> {MET_ROOT}/seed_*/")
print(f"aggregate -> {AGG_DIR}")


threshold check, how does threshold affect label_unanimity

In [ ]:
from sklearn.metrics import (precision_score, recall_score, fbeta_score,
                             f1_score, accuracy_score)

THRESHOLD_DIR = os.path.join(XRV_SAVE_DIR, "threshold_sweep")
os.makedirs(THRESHOLD_DIR, exist_ok=True)

THRESHOLDS = np.round(np.arange(0.20, 0.81, 0.05), 2)
TAGS = ["best", "epoch25"]

rows = []
for tag in TAGS:
    for s in HPARAMS2["seeds"]:
        rn = f"{HPARAMS2['run_name']}_seed{s}"
        f = os.path.join(XRV_SAVE_DIR, f"{rn}_{tag}_test_probs.npz")
        if not os.path.exists(f):
            continue
        d = np.load(f)
        yt, yp = d["y_true"], d["y_prob"]
        for thr in THRESHOLDS:
            pred = (yp >= thr).astype(int)
            rows.append({
                "checkpoint": tag, "seed": s, "thr": float(thr),
                "accuracy":  accuracy_score(yt, pred),
                "precision": precision_score(yt, pred, zero_division=0),
                "recall":    recall_score(yt, pred, zero_division=0),
                "f1":        f1_score(yt, pred, zero_division=0),
                "f2":        fbeta_score(yt, pred, beta=2, zero_division=0),
            })

if not rows:
    print("No *_test_probs.npz files found")
else:
    tdf = pd.DataFrame(rows)
    tdf.to_csv(os.path.join(THRESHOLD_DIR, "per_seed.csv"), index=False)

    for tag in TAGS:
        sub = tdf[tdf.checkpoint == tag]
        if sub.empty:
            continue
        print(f"Threshold Sweep: {tag.capitalize()}  ({sub.seed.nunique()} seeds)")
        agg = (sub.groupby("thr")[["accuracy", "precision", "recall", "f1", "f2"]]
               .agg(["mean", "std"]).round(4))
        print(agg.to_string())

        best_f2 = sub.groupby("thr").f2.mean().idxmax()
        print(f"\n  highest mean F2 at thr {best_f2:.2f} "
              f"(sensitivity only: do not report this as a tuned threshold)")

    agg_out = (tdf.groupby(["checkpoint", "thr"])
               [["accuracy", "precision", "recall", "f1", "f2"]]
               .agg(["mean", "std"]))
    agg_out.to_csv(os.path.join(THRESHOLD_DIR, "threshold_sweep_summary.csv"))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for tag, col in [("best", "steelblue"), ("epoch25", "indianred")]:
        sub = tdf[tdf.checkpoint == tag]
        if sub.empty:
            continue
        g = sub.groupby("thr")[["precision", "recall", "f2"]].mean()
        ax[0].plot(g.index, g.precision, color=col, ls="--",
                   label=f"{tag} precision")
        ax[0].plot(g.index, g.recall, color=col, label=f"{tag} recall")
        ax[1].plot(g.index, g.f2, color=col, label=f"{tag} F2")

    ax[0].axvline(0.5, color="grey", ls=":", lw=1)
    ax[0].set_xlabel("threshold"); ax[0].set_title("precision / recall")
    ax[0].legend(fontsize=8)
    ax[1].axvline(0.5, color="grey", ls=":", lw=1)
    ax[1].set_xlabel("threshold"); ax[1].set_title("F2")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(THRESHOLD_DIR, "threshold_sweep.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nsaved to {THRESHOLD_DIR}")


best vs epoch 25: exploratory

In [ ]:
import os, json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr, levene
import matplotlib.pyplot as plt

BEST_AGG = os.path.join(XRV_SAVE_DIR, "gradcam_metrics_best", "_aggregate")
E25_AGG  = os.path.join(XRV_SAVE_DIR, "gradcam_metrics_25", "_aggregate")
OUT_DIR  = os.path.join(XRV_SAVE_DIR, "checkpoint_comparison")
os.makedirs(OUT_DIR, exist_ok=True)

N_BOOT = 1000
rng = np.random.default_rng(42)

b = pd.read_csv(os.path.join(BEST_AGG, "per_seed_summary.csv"))
e = pd.read_csv(os.path.join(E25_AGG,  "per_seed_summary.csv"))

# which epoch each seed's "best" actually came from
epochs = []
for s in b.seed:
    p = os.path.join(XRV_SAVE_DIR,
                     f"{HPARAMS2['run_name']}_seed{s}_best_metrics.json")
    epochs.append(json.load(open(p)).get("best_epoch") if os.path.exists(p) else np.nan)
b = b.assign(best_epoch=epochs)

m = b.merge(e, on="seed", suffixes=("_best", "_e25"))
m["d_auprc"] = m.auprc_mean_best - m.auprc_mean_e25
m["d_point"] = m.pointing_hit_best - m.pointing_hit_e25
m.to_csv(os.path.join(OUT_DIR, "paired_per_seed.csv"), index=False)

print("Best vs Epoch25, paired over 21 seeds (Exploratory, §7 out of scope)")
print(m[["seed", "best_epoch", "auprc_mean_best", "auprc_mean_e25", "d_auprc",
         "pointing_hit_best", "pointing_hit_e25"]].round(4).to_string(index=False))

print(f"\n  best epoch: median {np.nanmedian(m.best_epoch):.0f}, "
      f"range {np.nanmin(m.best_epoch):.0f}-{np.nanmax(m.best_epoch):.0f}  "
      f"<- the selection confound")

print("is one checkpoint better on average?")
for name, cb, ce in [("AUPRC", "auprc_mean_best", "auprc_mean_e25"),
                     ("pointing", "pointing_hit_best", "pointing_hit_e25")]:
    diff = m[cb] - m[ce]
    stat, p = wilcoxon(m[cb], m[ce])
    boot = np.array([rng.choice(diff.values, len(diff), replace=True).mean()
                     for _ in range(N_BOOT)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"  {name:<9s} best {m[cb].mean():.4f}   ep25 {m[ce].mean():.4f}   "
          f"diff {diff.mean():+.4f} [{lo:+.4f}, {hi:+.4f}]   "
          f"Wilcoxon p={p:.4f}")

#  spread
print("does seed-induced variance differ between checkpoints?")
for name, cb, ce in [("AUPRC", "auprc_mean_best", "auprc_mean_e25"),
                     ("pointing", "pointing_hit_best", "pointing_hit_e25")]:
    sb, se = m[cb].std(ddof=1), m[ce].std(ddof=1)
    boot = np.array([
        (pd.Series(rng.choice(m[cb].values, len(m), replace=True)).std(ddof=1)
         - pd.Series(rng.choice(m[ce].values, len(m), replace=True)).std(ddof=1))
        for _ in range(N_BOOT)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    _, p = levene(m[cb], m[ce])
    print(f"  {name:<9s} SD best {sb:.4f}   SD ep25 {se:.4f}   "
          f"diff {sb - se:+.4f} [{lo:+.4f}, {hi:+.4f}]   Levene p={p:.4f}")

print("\n  If the best-checkpoint SD is larger, part of that is the selection")
print("  confound above, not extra seed sensitivity.")

# performance of same seeds comparison
rho_a, p_a = spearmanr(m.auprc_mean_best, m.auprc_mean_e25)
rho_p, p_p = spearmanr(m.pointing_hit_best, m.pointing_hit_e25)
print("seed consistency: is a 'good' seed good at both checkpoints?")
print(f"  AUPRC    Spearman rho {rho_a:+.4f} (p={p_a:.4f})")
print(f"  pointing Spearman rho {rho_p:+.4f} (p={p_p:.4f})")
print("\n  High rho -> seed quality is a stable property of the seed.")
print("  Low rho  -> it is epoch-specific noise, and 'a good seed' does not")
print("              transfer across checkpoints.")

# per image comparison
pb = pd.read_csv(os.path.join(BEST_AGG, "per_image_per_seed_scores.csv"))
pe = pd.read_csv(os.path.join(E25_AGG,  "per_image_per_seed_scores.csv"))
ib = pb.groupby("patientId").auprc.agg(["mean", "std"])
ie = pe.groupby("patientId").auprc.agg(["mean", "std"])
joined = ib.join(ie, lsuffix="_best", rsuffix="_e25", how="inner")
joined.to_csv(os.path.join(OUT_DIR, "paired_per_image.csv"))

rho_m, _ = spearmanr(joined["mean_best"], joined["mean_e25"])
rho_s, _ = spearmanr(joined["std_best"], joined["std_e25"])
print(f"Per-image ({len(joined)} images)")
print(f"  image difficulty  correlates rho {rho_m:+.4f}")
print(f"  image instability correlates rho {rho_s:+.4f}")
print(f"  median per-image SD: best {joined['std_best'].median():.4f}   "
      f"ep25 {joined['std_e25'].median():.4f}")

# graphs/figures
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))

x = np.arange(len(m))
ax[0].bar(x - 0.2, m.auprc_mean_best, 0.4, label="best", color="steelblue")
ax[0].bar(x + 0.2, m.auprc_mean_e25, 0.4, label="epoch25", color="indianred")
ax[0].set_xticks(x); ax[0].set_xticklabels(m.seed, rotation=90, fontsize=7)
ax[0].set_ylabel("mean AUPRC"); ax[0].set_title("per-seed AUPRC"); ax[0].legend(fontsize=8)

for i, r in m.iterrows():
    ax[1].plot([0, 1], [r.auprc_mean_best, r.auprc_mean_e25],
               color="grey", alpha=0.5, marker="o", ms=3)
ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["best", "epoch25"])
ax[1].set_ylabel("mean AUPRC"); ax[1].set_title("paired shift per seed")

ax[2].scatter(m.auprc_mean_best, m.auprc_mean_e25, s=30, color="seagreen")
lim = [min(m.auprc_mean_best.min(), m.auprc_mean_e25.min()) - 0.01,
       max(m.auprc_mean_best.max(), m.auprc_mean_e25.max()) + 0.01]
ax[2].plot(lim, lim, "k--", lw=1)
ax[2].set_xlabel("best"); ax[2].set_ylabel("epoch25")
ax[2].set_title(f"seed consistency (rho {rho_a:+.2f})")

ax[3].scatter(m.best_epoch, m.auprc_mean_best, s=30, color="darkorange")
ax[3].set_xlabel("epoch the best checkpoint came from")
ax[3].set_ylabel("mean AUPRC (best)")
ax[3].set_title("the selection confound")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "best_vs_epoch25.png"), dpi=150,
            bbox_inches="tight")
plt.show()

summary = {
    "status": "exploratory, §7 puts checkpoint comparison out of scope",
    "n_seeds": len(m),
    "best_epoch": {"median": float(np.nanmedian(m.best_epoch)),
                   "min": float(np.nanmin(m.best_epoch)),
                   "max": float(np.nanmax(m.best_epoch))},
    "auprc": {
        "mean_best": float(m.auprc_mean_best.mean()),
        "mean_e25": float(m.auprc_mean_e25.mean()),
        "sd_best": float(m.auprc_mean_best.std(ddof=1)),
        "sd_e25": float(m.auprc_mean_e25.std(ddof=1)),
        "seed_rank_rho": float(rho_a),
    },
    "pointing": {
        "mean_best": float(m.pointing_hit_best.mean()),
        "mean_e25": float(m.pointing_hit_e25.mean()),
        "sd_best": float(m.pointing_hit_best.std(ddof=1)),
        "sd_e25": float(m.pointing_hit_e25.std(ddof=1)),
        "seed_rank_rho": float(rho_p),
    },
    "per_image": {"difficulty_rho": float(rho_m), "instability_rho": float(rho_s)},
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print(f"\nsaved -> {OUT_DIR}")


how many pos vs neg test set

In [ ]:
import pandas as pd
import os

# Load the split and labels
test_df = pd.read_csv(os.path.join(EXTRACT_DIR, "test_split.csv"))
labels_df = pd.read_csv(os.path.join(EXTRACT_DIR, "stage_2_train_labels.csv"))

# Get unique patient IDs in the test split
test_patients = set(test_df["patientId"])

# Keep labels belonging to the test split
test_labels = labels_df[labels_df["patientId"].isin(test_patients)]

# Count unique patients by Target
patient_targets = test_labels.groupby("patientId")["Target"].max()

positive = (patient_targets == 1).sum()
negative = (patient_targets == 0).sum()

print("Test patients:", len(patient_targets))
print("Positive patients:", positive)
print("Negative patients:", negative)
print("Positive percentage:", positive / len(patient_targets) * 100)


fixed rel-u

In [ ]:
# Grad-CAM export loop for epoch 25 checkpoints

CKPT_TAG = "epoch25"
FOLDER_25 = "gradcam_npy_25_fixedrelu"
CAM_ROOT = os.path.join(XRV_SAVE_DIR, FOLDER_25)

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")
box_patients = set(
    labels_df[(labels_df["Target"] == 1) & (labels_df["width"].notna())]["patientId"].tolist()
)
box_test_df = (
    test_ds[test_ds["patientId"].isin(box_patients)]
    .drop_duplicates(subset="patientId")     # one row per patient, not per box
    .reset_index(drop=True)
)
print(f"Box-bearing test patients: {len(box_test_df)}")

gradcam_root_25 = os.path.join(XRV_SAVE_DIR, FOLDER_25)
os.makedirs(gradcam_root_25, exist_ok=True)
print(f"Root: {gradcam_root_25}")

for seed in HPARAMS2["seeds"]:
    run_name = f"{HPARAMS2['run_name']}_seed{seed}"
    ckpt_path = os.path.join(XRV_SAVE_DIR, f"{run_name}_{CKPT_TAG}.pth")

    if not os.path.exists(ckpt_path):
        print(f"[seed {seed}] {CKPT_TAG} checkpoint missing, skipping")
        continue

    seed_out = os.path.join(gradcam_root_25, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)

    model_gc = XRVClassifier(HPARAMS2["xrv_weights"], HPARAMS2["dropout"]).to(device)
    state_dict_gc = torch.load(ckpt_path, map_location="cpu")
    missing_keys, unexpected_keys = model_gc.load_state_dict(state_dict_gc, strict=True)
    print(f"[seed {seed}] strict=True load ({CKPT_TAG})")
    print(f"  missing keys    : {missing_keys}")
    print(f"  unexpected keys : {unexpected_keys}")
    model_gc.eval()

    gradcam_model_gc = GradCAM(model_gc).to(device)
    gradcam_model_gc.eval()

    preds_rows = []

    for _, row in tqdm(box_test_df.iterrows(), total=len(box_test_df),
                       desc=f"GradCAM ep25 seed {seed}"):
        patient_id = row["patientId"]
        if patient_id not in npy_map:
            continue

        img_np = np.load(npy_map[patient_id]).astype(np.float32)
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        img_np = val_transform_xrv(image=img_np)["image"]
        img_np = img_np.astype(np.float32)
        img_np = xrv.datasets.normalize(img_np, 255)
        img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).float().to(device)

        logits = gradcam_model_gc(img_tensor)
        prob = torch.softmax(logits.float(), dim=1)[0, 1].item()
        pred = int(prob >= 0.5)

        gradcam_model_gc.zero_grad()
        logits[0, 1].backward()

        gradients = gradcam_model_gc.get_activations_gradient()            # [1, C, 7, 7]
        pooled_g  = gradients.mean(dim=[0, 2, 3])                          # [C]
        activs    = gradcam_model_gc.get_activations(img_tensor).detach()  # [1, C, 7, 7]
        weighted  = activs * pooled_g.view(1, -1, 1, 1)
        raw_cam   = weighted.mean(dim=1).squeeze().cpu().numpy()           # raw 7x7, no ReLU

        np.save(os.path.join(seed_out, f"{patient_id}.npy"), raw_cam)

        preds_rows.append({
            "seed": seed,
            "patientId": patient_id,
            "true_label": int(row["Target"]),
            "pred_label": pred,
            "pred_prob": round(prob, 6),
        })

    pd.DataFrame(preds_rows).to_csv(os.path.join(seed_out, "preds.csv"), index=False)
    print(f"[seed {seed}] {len(preds_rows)} CAMs -> {seed_out}")

    del model_gc, gradcam_model_gc
    torch.cuda.empty_cache()

print(f"Grad-CAM export complete ({CKPT_TAG}).")


In [ ]:
import os, json, itertools
import numpy as np
import pandas as pd
import cv2
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

CKPT_TAG   = "epoch25"
CAM_ROOT   = os.path.join(XRV_SAVE_DIR, "gradcam_npy_25_fixedrelu")
MET_ROOT   = os.path.join(XRV_SAVE_DIR, "gradcam_metrics_25_fixedrelu")
AGG_DIR    = os.path.join(MET_ROOT, "_aggregate")
os.makedirs(AGG_DIR, exist_ok=True)

SCORE_SIZE = 1024        # §5
CAM_NATIVE = 7           # raw map resolution
N_BOOT     = 1000        # §6.2 / §6.3
BOOT_SEED  = 42

ARUN = {"XRAI": 0.52, "GCAM": 0.41, "GRAD": 0.34,
        "IG": 0.32, "GGCAM": 0.32, "SG": 0.17, "SIG": 0.16}
ARUN_ORDER = ["XRAI", "GCAM", "GRAD", "IG", "GGCAM", "SG", "SIG"]

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")

# helpers
def load_raw(seed, pid):

    f = os.path.join(CAM_ROOT, f"seed_{seed}", f"{pid}.npy")
    if not os.path.exists(f):
        return None
    cam = np.load(f).astype(np.float32)
    # §5 amendment: rectify CAM before normalization
    cam = np.maximum(cam, 0)
    # Max-normalize the non-negative CAM to [0,1]
    max_cam = cam.max()
    if max_cam > 0:
        cam = cam / max_cam
    else:
        cam = np.zeros_like(cam)
    return cam

def upsample(cam):
    return cv2.resize(cam, (SCORE_SIZE, SCORE_SIZE), interpolation=cv2.INTER_LINEAR)

def box_mask(pid):

    r = labels_df[(labels_df.patientId == pid) & (labels_df.Target == 1)]
    r = r.dropna(subset=["x", "y", "width", "height"])
    if r.empty:
        return None
    m = np.zeros((SCORE_SIZE, SCORE_SIZE), np.uint8)
    for _, b in r.iterrows():
        x0, y0 = int(b.x), int(b.y)
        x1, y1 = int(b.x + b.width), int(b.y + b.height)
        m[max(y0, 0):min(y1, SCORE_SIZE), max(x0, 0):min(x1, SCORE_SIZE)] = 1
    return m if m.sum() else None

def auprc(mask_flat, map_flat):

    if mask_flat.sum() == 0:
        return np.nan
    return float(average_precision_score(mask_flat, map_flat))

# whats on the disk
pred_paths = {s: os.path.join(CAM_ROOT, f"seed_{s}", "preds.csv")
              for s in HPARAMS2["seeds"]}
pred_paths = {s: p for s, p in pred_paths.items() if os.path.exists(p)}
SEEDS = sorted(pred_paths.keys())

preds = pd.concat([pd.read_csv(p) for p in pred_paths.values()])
patients = sorted(preds.patientId.unique())
masks = {pid: box_mask(pid) for pid in patients}
masks = {k: v for k, v in masks.items() if v is not None}

print(f"[{CKPT_TAG}] {len(SEEDS)} seeds, {len(patients)} patients, "
      f"{len(masks)} with usable boxes")
missing_seeds = sorted(set(HPARAMS2["seeds"]) - set(SEEDS))
if missing_seeds:
    print(f"  [!] missing seeds {missing_seeds} -- §6.5 requires documenting these")

# score every (seed, image)
rows, zero_maps = [], []

for i, (pid, gt) in enumerate(masks.items(), 1):
    gt_flat = gt.reshape(-1)
    prevalence = float(gt.mean())          # exact AUPRC of a random ranker
    for s in SEEDS:
        raw = load_raw(s, pid)
        if raw is None:
            continue
        if raw.max() <= 0:                 # §6.5
            zero_maps.append({"seed": s, "patientId": pid})
            rows.append({"seed": s, "patientId": pid, "auprc": 0.0,
                         "pointing_hit": 0, "prevalence": prevalence,
                         "all_zero": True})
            continue
        m = upsample(raw)
        py, px = np.unravel_index(np.argmax(m), m.shape)
        rows.append({"seed": s, "patientId": pid,
                     "auprc": auprc(gt_flat, m.reshape(-1)),
                     "pointing_hit": int(gt[py, px] == 1),
                     "prevalence": prevalence,
                     "all_zero": False})
    if i % 50 == 0:
        print(f"  {i}/{len(masks)} images scored")

sdf = pd.DataFrame(rows)
sdf.to_csv(os.path.join(AGG_DIR, "per_image_per_seed_scores.csv"), index=False)
print(f"\nscored {len(sdf)} (seed, image) pairs; {len(zero_maps)} all-zero maps")

# per seed output
per_seed_rows = []

for seed in SEEDS:
    sub = sdf[sdf.seed == seed]
    preds_s = pd.read_csv(pred_paths[seed])[["patientId", "pred_prob"]]
    merged = sub.merge(preds_s, on="patientId", how="left")

    seed_out = os.path.join(MET_ROOT, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)
    merged.to_csv(os.path.join(seed_out, "localization.csv"), index=False)

    pos = merged[merged.pred_prob >= 0.5]
    s_sum = {
        "seed": seed, "checkpoint": CKPT_TAG, "n_images": len(merged),
        "auprc_mean": float(merged.auprc.mean()),
        "auprc_sd_across_images": float(merged.auprc.std()),
        "auprc_median": float(merged.auprc.median()),
        "pointing_hit": float(merged.pointing_hit.mean()),
        "auprc_pred_positive": float(pos.auprc.mean()) if len(pos) else None,
        "n_pred_positive": int(len(pos)),
        "mean_pred_prob": float(merged.pred_prob.mean()),
    }
    with open(os.path.join(seed_out, "summary.json"), "w") as f:
        json.dump(s_sum, f, indent=4)

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].hist(merged.auprc, bins=30, color="indianred")
    ax[0].axvline(merged.prevalence.mean(), color="r", ls="--",
                  label=f"random {merged.prevalence.mean():.3f}")
    ax[0].set_title(f"seed {seed} (ep25): per-image AUPRC")
    ax[0].set_xlabel("AUPRC"); ax[0].legend(fontsize=8)
    ax[1].hist(merged.pred_prob, bins=25, color="slategray")
    ax[1].axvline(0.5, color="r", ls="--")
    ax[1].set_title(f"seed {seed} (ep25): p(pneumonia)"); ax[1].set_xlabel("prob")
    plt.tight_layout()
    plt.savefig(os.path.join(seed_out, "metrics.png"), dpi=140, bbox_inches="tight")
    plt.close()

    per_seed_rows.append(s_sum)
    print(f"  seed {seed:>5}: auprc {s_sum['auprc_mean']:.4f}  "
          f"pointing {s_sum['pointing_hit']:.4f}")

psdf = pd.DataFrame(per_seed_rows)
psdf.to_csv(os.path.join(AGG_DIR, "per_seed_summary.csv"), index=False)

# per-image variance
per_img = sdf.groupby("patientId").auprc.agg(["mean", "std"])
q1, med, q3 = per_img["std"].quantile([0.25, 0.50, 0.75])
per_img.to_csv(os.path.join(AGG_DIR, "per_image_variance.csv"))

print("6.1  per-image variance  (SD of AUPRC across seeds, per image)")
print(f"  median SD : {med:.4f}")
print(f"  IQR       : [{q1:.4f}, {q3:.4f}]")
print(f"  min / max : {per_img['std'].min():.4f} / {per_img['std'].max():.4f}")

# C1: spread vs gaps
per_seed = psdf.set_index("seed").auprc_mean
spread = float(per_seed.std(ddof=1))

rng = np.random.default_rng(BOOT_SEED)
boot = np.array([pd.Series(rng.choice(per_seed.values, len(per_seed), replace=True))
                 .std(ddof=1) for _ in range(N_BOOT)])
lo, hi = np.percentile(boot, [2.5, 97.5])

gaps = {f"{ARUN_ORDER[i]}->{ARUN_ORDER[i+1]}":
        round(ARUN[ARUN_ORDER[i]] - ARUN[ARUN_ORDER[i+1]], 4)
        for i in range(len(ARUN_ORDER) - 1)}

yy, xx = np.mgrid[0:SCORE_SIZE, 0:SCORE_SIZE]
c = SCORE_SIZE / 2
centre_map = np.exp(-((xx - c) ** 2 + (yy - c) ** 2) / (2 * (SCORE_SIZE / 5) ** 2))
centre_flat = centre_map.astype(np.float32).reshape(-1)
centre_scores = [auprc(gt.reshape(-1), centre_flat) for gt in masks.values()]

print("C1 test: seed-induced spread vs published adjacent gaps")
print(psdf[["seed", "auprc_mean", "pointing_hit", "auprc_pred_positive"]]
      .round(4).to_string(index=False))
print(f"\n  mean AUPRC across seeds  : {per_seed.mean():.4f}")
print(f"  per-seed range           : {per_seed.min():.4f} - {per_seed.max():.4f}")
print(f"  Spread (SD of per-seed)  : {spread:.4f}")
print(f"  bootstrap 95% CI         : [{lo:.4f}, {hi:.4f}]")
print(f"\n  random-ranker baseline   : {sdf.prevalence.mean():.4f}")
print(f"  centred-Gaussian baseline: {np.mean(centre_scores):.4f}")
print(f"  Arun et al. GCAM         : {ARUN['GCAM']:.2f}")
print("\n  adjacent gap            gap     spread >= gap?")
for name, g in gaps.items():
    print(f"    {name:<22s} {g:.2f}    {'yes' if spread >= g else 'no'}")

n_ge = sum(spread >= g for g in gaps.values())
print(f"\n  spread matches or exceeds {n_ge}/{len(gaps)} adjacent gaps")
print(f"  H1 {'supported' if n_ge > 0 else 'not supported'} "
      f"(unsupported iff spread is small relative to every gap)")

# C2: ranking flips

pairs_rank = [(ARUN_ORDER[i], ARUN_ORDER[i + 1]) for i in range(len(ARUN_ORDER) - 1)]
flip_rows = []

for s in SEEDS:
    scores = dict(ARUN)
    scores["GCAM"] = float(per_seed[s])
    flips = {f"{a}->{b}": int((ARUN[a] > ARUN[b]) != (scores[a] > scores[b]))
             for a, b in pairs_rank}
    new_rank = sorted(ARUN_ORDER, key=lambda m: -scores[m]).index("GCAM") + 1
    flip_rows.append({"seed": s, "gcam_score": scores["GCAM"],
                      "new_rank": new_rank, "rank_shift": new_rank - 2,
                      "n_flips": sum(flips.values()), **flips})

fdf = pd.DataFrame(flip_rows)
fdf.to_csv(os.path.join(AGG_DIR, "c2_flip_matrix.csv"), index=False)

seeds_ge1 = int((fdf.n_flips >= 1).sum())
seeds_ge2 = int((fdf.n_flips >= 2).sum())

print("6.3  C2 test: ranking flips")
print(fdf[["seed", "gcam_score", "new_rank", "rank_shift", "n_flips"]]
      .round(4).to_string(index=False))
print(f"\n  published GCAM rank: 2 of {len(ARUN_ORDER)}")
print(f"  seeds with >=1 adjacent-pair flip : {seeds_ge1}/{len(SEEDS)}")
print(f"  seeds with >=2 adjacent-pair flips: {seeds_ge2}/{len(SEEDS)}")
print(f"  seeds whose GCAM rank changes     : {int((fdf.rank_shift != 0).sum())}"
      f"/{len(SEEDS)}")
print(f"\n  H2 criterion (>=2 pairs in >=3 seeds): "
      f"{'met' if seeds_ge2 >= 3 else 'not met'}")
print("  [see code note: the >=2-pairs criterion may be unreachable by design]")

boot_rank = np.array([
    (pd.Series(rng.choice(fdf.rank_shift.values, len(fdf), replace=True)) != 0).mean()
    for _ in range(N_BOOT)])
rlo, rhi = np.percentile(boot_rank, [2.5, 97.5])
print(f"  rank-change rate {float((fdf.rank_shift != 0).mean()):.3f} "
      f"(bootstrap 95% CI [{rlo:.3f}, {rhi:.3f}])")

# 6.4 model-equivalence control
auc_rows = []
for s in SEEDS:
    rn = f"{HPARAMS2['run_name']}_seed{s}"
    p = os.path.join(XRV_SAVE_DIR, f"{rn}_test_metrics.json")
    if not os.path.exists(p):
        continue
    d = json.load(open(p))
    blk = d.get("epoch25") if isinstance(d.get("epoch25"), dict) else d
    auc_rows.append({"seed": s,
                     "test_roc_auc": blk.get("test_roc_auc"),
                     "test_pr_auc": blk.get("test_pr_auc"),
                     "best_epoch": d.get("best_epoch")})

if auc_rows:
    mdf = pd.DataFrame(auc_rows)
    mdf.to_csv(os.path.join(AGG_DIR, "model_equivalence.csv"), index=False)
    print("6.4  model-equivalence control (epoch-25 checkpoints)")
    print(mdf.round(4).to_string(index=False))
    print(f"\n  test ROC-AUC {mdf.test_roc_auc.mean():.4f} "
          f"+/- {mdf.test_roc_auc.std():.4f}")
    print(f"  test PR-AUC  {mdf.test_pr_auc.mean():.4f} "
          f"+/- {mdf.test_pr_auc.std():.4f}")

# exploratory: inter-seed agreement, computed at 7x7
pairs = list(itertools.combinations(SEEDS, 2))
agr_rows = []
px_per_cell = SCORE_SIZE / CAM_NATIVE

for pid in masks:
    cams = {s: load_raw(s, pid) for s in SEEDS}
    cams = {s: c for s, c in cams.items() if c is not None}
    if len(cams) < 2:
        continue
    for a, b in pairs:
        if a not in cams or b not in cams:
            continue
        rho, _ = spearmanr(cams[a].ravel(), cams[b].ravel())
        ya, xa = np.unravel_index(np.argmax(cams[a]), cams[a].shape)
        yb, xb = np.unravel_index(np.argmax(cams[b]), cams[b].shape)
        agr_rows.append({"patientId": pid, "seed_a": a, "seed_b": b,
                         "spearman": float(rho),
                         "peak_px": float(np.hypot(ya - yb, xa - xb) * px_per_cell)})

adf = pd.DataFrame(agr_rows)
adf.to_csv(os.path.join(AGG_DIR, "cam_agreement.csv"), index=False)

wide = preds.pivot_table(index="patientId", columns="seed", values="pred_label")
unanimous = float((wide.nunique(axis=1) == 1).mean())

print(f"Exploratory: inter-seed agreement ({len(pairs)} pairs, at 7x7 native)")
print(f"  spearman  {adf.spearman.mean():.4f} +/- {adf.spearman.std():.4f}")
print(f"  peak_px   {adf.peak_px.mean():.1f} +/- {adf.peak_px.std():.1f} "
      f"(1024-px units)")
print(f"  seeds unanimous on the label (thr 0.5): {unanimous:.1%}")
print(f"  mean prob SD across seeds: "
      f"{preds.groupby('patientId').pred_prob.std().mean():.4f}")

# save
summary = {
    "status": "confirmatory (epoch-25 final checkpoint, per §5)",
    "plan_version": "v2 2026-08-31",
    "checkpoint": CKPT_TAG, "score_size": SCORE_SIZE,
    "n_seeds": len(SEEDS), "n_images": len(masks),
    "missing_seeds": missing_seeds,
    "n_all_zero_maps": len(zero_maps),
    "primary_auprc": {"mean_across_seeds": float(per_seed.mean()),
                      "spread_sd_of_per_seed_mean": spread,
                      "bootstrap_ci_95": [float(lo), float(hi)],
                      "per_seed_min": float(per_seed.min()),
                      "per_seed_max": float(per_seed.max())},
    "per_image_sd": {"median": float(med), "iqr": [float(q1), float(q3)]},
    "baseline_random": float(sdf.prevalence.mean()),
    "baseline_centre_gaussian": float(np.mean(centre_scores)),
    "arun_gaps": gaps,
    "n_gaps_matched_or_exceeded": n_ge,
    "c2": {"seeds_ge1_flip": seeds_ge1, "seeds_ge2_flips": seeds_ge2,
           "seeds_rank_changed": int((fdf.rank_shift != 0).sum()),
           "criterion_met": bool(seeds_ge2 >= 3)},
    "pointing_hit": {"mean": float(psdf.pointing_hit.mean()),
                     "std": float(psdf.pointing_hit.std())},
    "agreement_exploratory": {
        "spearman": {"mean": float(adf.spearman.mean()),
                     "std": float(adf.spearman.std())},
        "peak_px": {"mean": float(adf.peak_px.mean()),
                    "std": float(adf.peak_px.std())}},
    "label_unanimity": unanimous,
}
with open(os.path.join(AGG_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].bar(range(len(psdf)), psdf.auprc_mean, color="indianred")
ax[0].axhline(ARUN["GCAM"], color="k", ls="--", label="Arun GCAM 0.41")
ax[0].axhline(sdf.prevalence.mean(), color="r", ls=":", label="random baseline")
ax[0].set_xticks(range(len(psdf)))
ax[0].set_xticklabels(psdf.seed, rotation=90, fontsize=7)
ax[0].set_ylabel("mean AUPRC"); ax[0].set_title("per-seed AUPRC (ep25)")
ax[0].legend(fontsize=8)

ax[1].hist(per_img["std"], bins=40, color="seagreen")
ax[1].axvline(med, color="r", ls="--", label=f"median {med:.3f}")
ax[1].set_xlabel("per-image SD across seeds")
ax[1].set_title("6.1 per-image variance"); ax[1].legend(fontsize=8)

ax[2].bar(range(len(gaps)), list(gaps.values()), color="lightgrey",
          label="Arun adjacent gap")
ax[2].axhline(spread, color="crimson", lw=2, label=f"seed spread {spread:.3f}")
ax[2].fill_between([-0.5, len(gaps) - 0.5], lo, hi, color="crimson", alpha=0.15,
                   label="95% CI")
ax[2].set_xticks(range(len(gaps)))
ax[2].set_xticklabels(gaps.keys(), rotation=45, ha="right", fontsize=7)
ax[2].set_title("6.2 C1: spread vs gaps"); ax[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(AGG_DIR, "metrics.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nper-seed  -> {MET_ROOT}/seed_*/")
print(f"aggregate -> {AGG_DIR}")


best fixed relu

In [ ]:
# Grad-CAM export loop for epoch 25 checkpoints

CKPT_TAG = "best"
FOLDER_25 = "gradcam_npy_best_fixedrelu"
CAM_ROOT = os.path.join(XRV_SAVE_DIR, "gradcam_npy_best_fixedrelu")

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")
box_patients = set(
    labels_df[(labels_df["Target"] == 1) & (labels_df["width"].notna())]["patientId"].tolist()
)
box_test_df = (
    test_ds[test_ds["patientId"].isin(box_patients)]
    .drop_duplicates(subset="patientId")     # one row per patient, not per box
    .reset_index(drop=True)
)
print(f"Box-bearing test patients: {len(box_test_df)}")

gradcam_root_25 = os.path.join(XRV_SAVE_DIR, FOLDER_25)
os.makedirs(gradcam_root_25, exist_ok=True)
print(f"Root: {gradcam_root_25}")

for seed in HPARAMS2["seeds"]:
    run_name = f"{HPARAMS2['run_name']}_seed{seed}"
    ckpt_path = os.path.join(XRV_SAVE_DIR, f"{run_name}_{CKPT_TAG}.pth")

    if not os.path.exists(ckpt_path):
        print(f"[seed {seed}] {CKPT_TAG} checkpoint missing, skipping")
        continue

    seed_out = os.path.join(gradcam_root_25, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)

    model_gc = XRVClassifier(HPARAMS2["xrv_weights"], HPARAMS2["dropout"]).to(device)
    state_dict_gc = torch.load(ckpt_path, map_location="cpu")
    missing_keys, unexpected_keys = model_gc.load_state_dict(state_dict_gc, strict=True)
    print(f"[seed {seed}] strict=True load ({CKPT_TAG})")
    print(f"  missing keys    : {missing_keys}")
    print(f"  unexpected keys : {unexpected_keys}")
    model_gc.eval()

    gradcam_model_gc = GradCAM(model_gc).to(device)
    gradcam_model_gc.eval()

    preds_rows = []

    for _, row in tqdm(box_test_df.iterrows(), total=len(box_test_df),
                       desc=f"GradCAM ep25 seed {seed}"):
        patient_id = row["patientId"]
        if patient_id not in npy_map:
            continue

        img_np = np.load(npy_map[patient_id]).astype(np.float32)
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        img_np = val_transform_xrv(image=img_np)["image"]
        img_np = img_np.astype(np.float32)
        img_np = xrv.datasets.normalize(img_np, 255)
        img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).float().to(device)

        logits = gradcam_model_gc(img_tensor)
        prob = torch.softmax(logits.float(), dim=1)[0, 1].item()
        pred = int(prob >= 0.5)

        gradcam_model_gc.zero_grad()
        logits[0, 1].backward()

        gradients = gradcam_model_gc.get_activations_gradient()            # [1, C, 7, 7]
        pooled_g  = gradients.mean(dim=[0, 2, 3])                          # [C]
        activs    = gradcam_model_gc.get_activations(img_tensor).detach()  # [1, C, 7, 7]
        weighted  = activs * pooled_g.view(1, -1, 1, 1)
        raw_cam   = weighted.mean(dim=1).squeeze().cpu().numpy()           # raw 7x7, no ReLU

        np.save(os.path.join(seed_out, f"{patient_id}.npy"), raw_cam)

        preds_rows.append({
            "seed": seed,
            "patientId": patient_id,
            "true_label": int(row["Target"]),
            "pred_label": pred,
            "pred_prob": round(prob, 6),
        })

    pd.DataFrame(preds_rows).to_csv(os.path.join(seed_out, "preds.csv"), index=False)
    print(f"[seed {seed}] {len(preds_rows)} CAMs -> {seed_out}")

    del model_gc, gradcam_model_gc
    torch.cuda.empty_cache()

print(f"Grad-CAM export complete ({CKPT_TAG}).")


In [ ]:
import os, json, itertools
import numpy as np
import pandas as pd
import cv2
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

CKPT_TAG   = "best"                     # set "epoch25" to re-run the confirmatory arm
CAM_ROOT   = os.path.join(XRV_SAVE_DIR, f"gradcam_npy_{CKPT_TAG}_fixedrelu")
OUT_DIR    = os.path.join(XRV_SAVE_DIR, f"gradcam_metrics_{CKPT_TAG}_fixedrelu")
AGG_DIR    = os.path.join(OUT_DIR, "_aggregate")
os.makedirs(AGG_DIR, exist_ok=True)

SCORE_SIZE = 1024
CAM_NATIVE = 7           # raw map resolution; agreement is computed here
N_BOOT     = 1000
BOOT_SEED  = 42

labels_df = pd.read_csv("/content/rsna-pneumonia-dataset-npy/stage_2_train_labels.csv")

#helpers
def load_raw(seed, pid):

    f = os.path.join(CAM_ROOT, f"seed_{seed}", f"{pid}.npy")
    if not os.path.exists(f):
        return None
    cam = np.load(f).astype(np.float32)
    # §5 amendment: rectify CAM before normalization
    cam = np.maximum(cam, 0)
    # Max-normalize the non-negative CAM to [0,1]
    max_cam = cam.max()
    if max_cam > 0:
        cam = cam / max_cam
    else:
        cam = np.zeros_like(cam)

    return cam

def upsample(cam):
    return cv2.resize(cam, (SCORE_SIZE, SCORE_SIZE), interpolation=cv2.INTER_LINEAR)

def box_mask(pid):

    r = labels_df[(labels_df.patientId == pid) & (labels_df.Target == 1)]
    r = r.dropna(subset=["x", "y", "width", "height"])
    if r.empty:
        return None
    m = np.zeros((SCORE_SIZE, SCORE_SIZE), np.uint8)
    for _, b in r.iterrows():
        x0, y0 = int(b.x), int(b.y)
        x1, y1 = int(b.x + b.width), int(b.y + b.height)
        m[max(y0, 0):min(y1, SCORE_SIZE), max(x0, 0):min(x1, SCORE_SIZE)] = 1
    return m if m.sum() else None

def auprc(mask_flat, map_flat):
    if mask_flat.sum() == 0:
        return np.nan
    return float(average_precision_score(mask_flat, map_flat))

# whats on the disk
pred_paths = {s: os.path.join(CAM_ROOT, f"seed_{s}", "preds.csv")
              for s in HPARAMS2["seeds"]}
pred_paths = {s: p for s, p in pred_paths.items() if os.path.exists(p)}
SEEDS = sorted(pred_paths.keys())

preds = pd.concat([pd.read_csv(p) for p in pred_paths.values()])
patients = sorted(preds.patientId.unique())
masks = {pid: box_mask(pid) for pid in patients}
masks = {k: v for k, v in masks.items() if v is not None}

print(f"[{CKPT_TAG}] {len(SEEDS)} seeds, {len(patients)} patients, "
      f"{len(masks)} with usable boxes")
if len(SEEDS) < len(HPARAMS2["seeds"]):
    print(f"  [!] missing: {sorted(set(HPARAMS2['seeds']) - set(SEEDS))}")

# score every (seed, image)
rows, zero_maps = [], []

for i, (pid, gt) in enumerate(masks.items(), 1):
    gt_flat = gt.reshape(-1)
    prevalence = float(gt.mean())          # exact AUPRC of a random ranker
    for s in SEEDS:
        raw = load_raw(s, pid)
        if raw is None:
            continue
        if raw.max() <= 0:
            zero_maps.append({"seed": s, "patientId": pid})
            rows.append({"seed": s, "patientId": pid, "auprc": 0.0,
                         "pointing_hit": 0, "prevalence": prevalence,
                         "all_zero": True})
            continue
        m = upsample(raw)
        py, px = np.unravel_index(np.argmax(m), m.shape)
        rows.append({"seed": s, "patientId": pid,
                     "auprc": auprc(gt_flat, m.reshape(-1)),
                     "pointing_hit": int(gt[py, px] == 1),
                     "prevalence": prevalence,
                     "all_zero": False})
    if i % 50 == 0:
        print(f"  {i}/{len(masks)} images scored")

sdf = pd.DataFrame(rows)
sdf.to_csv(os.path.join(AGG_DIR, "per_image_per_seed_scores.csv"), index=False)
print(f"\nscored {len(sdf)} (seed, image) pairs; {len(zero_maps)} all-zero maps")

# per seed output
per_seed_rows = []

for seed in SEEDS:
    sub = sdf[sdf.seed == seed]
    preds_s = pd.read_csv(pred_paths[seed])[["patientId", "pred_prob"]]
    merged = sub.merge(preds_s, on="patientId", how="left")

    seed_out = os.path.join(OUT_DIR, f"seed_{seed}")
    os.makedirs(seed_out, exist_ok=True)
    merged.to_csv(os.path.join(seed_out, "localization.csv"), index=False)

    pos = merged[merged.pred_prob >= 0.5]
    s_sum = {
        "seed": seed, "checkpoint": CKPT_TAG, "n_images": len(merged),
        "auprc_mean": float(merged.auprc.mean()),
        "auprc_sd_across_images": float(merged.auprc.std()),
        "auprc_median": float(merged.auprc.median()),
        "pointing_hit": float(merged.pointing_hit.mean()),
        "auprc_pred_positive": float(pos.auprc.mean()) if len(pos) else None,
        "n_pred_positive": int(len(pos)),
        "mean_pred_prob": float(merged.pred_prob.mean()),
    }
    with open(os.path.join(seed_out, "summary.json"), "w") as f:
        json.dump(s_sum, f, indent=4)

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].hist(merged.auprc, bins=30, color="steelblue")
    ax[0].axvline(merged.prevalence.mean(), color="r", ls="--",
                  label=f"random {merged.prevalence.mean():.3f}")
    ax[0].set_title(f"seed {seed} ({CKPT_TAG}): per-image AUPRC")
    ax[0].set_xlabel("AUPRC"); ax[0].legend(fontsize=8)
    ax[1].hist(merged.pred_prob, bins=25, color="slategray")
    ax[1].axvline(0.5, color="r", ls="--")
    ax[1].set_title(f"seed {seed}: p(pneumonia)"); ax[1].set_xlabel("prob")
    plt.tight_layout()
    plt.savefig(os.path.join(seed_out, "metrics.png"), dpi=140, bbox_inches="tight")
    plt.close()

    per_seed_rows.append(s_sum)
    print(f"  seed {seed:>5}: auprc {s_sum['auprc_mean']:.4f}  "
          f"pointing {s_sum['pointing_hit']:.4f}")

psdf = pd.DataFrame(per_seed_rows)
psdf.to_csv(os.path.join(AGG_DIR, "per_seed_summary.csv"), index=False)

# aggregates
per_seed = psdf.set_index("seed").auprc_mean
spread = float(per_seed.std(ddof=1))          # SD of per-seed means

rng = np.random.default_rng(BOOT_SEED)
boot = np.array([pd.Series(rng.choice(per_seed.values, len(per_seed), replace=True))
                 .std(ddof=1) for _ in range(N_BOOT)])
lo, hi = np.percentile(boot, [2.5, 97.5])

per_img = sdf.groupby("patientId").auprc.agg(["mean", "std"])
q1, med, q3 = per_img["std"].quantile([0.25, 0.50, 0.75])

yy, xx = np.mgrid[0:SCORE_SIZE, 0:SCORE_SIZE]
c = SCORE_SIZE / 2
centre_map = np.exp(-((xx - c) ** 2 + (yy - c) ** 2) / (2 * (SCORE_SIZE / 5) ** 2))
centre_flat = centre_map.astype(np.float32).reshape(-1)
centre_scores = [auprc(gt.reshape(-1), centre_flat) for gt in masks.values()]

print(f"[{CKPT_TAG}] Primary: per-image AUPRC (threshold-free)")
print(psdf[["seed", "auprc_mean", "pointing_hit", "auprc_pred_positive"]]
      .round(4).to_string(index=False))
print(f"\n  mean across seeds        : {per_seed.mean():.4f}")
print(f"  Spread (SD of per-seed)  : {spread:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  per-seed range           : {per_seed.min():.4f} - {per_seed.max():.4f}")
print(f"\n  random-ranker baseline   : {sdf.prevalence.mean():.4f} "
      f"(= mean positive-pixel rate)")
print(f"  centred-Gaussian baseline: {np.mean(centre_scores):.4f}")
print(f"\n  per-image SD across seeds: median {med:.4f}  IQR [{q1:.4f}, {q3:.4f}]")
print(f"  pointing hit rate        : {psdf.pointing_hit.mean():.4f} "
      f"+/- {psdf.pointing_hit.std():.4f}")

# inter-seed agreement, computed at 7x7
pairs = list(itertools.combinations(SEEDS, 2))
agr_rows = []
px_per_cell = SCORE_SIZE / CAM_NATIVE

for pid in masks:
    cams = {s: load_raw(s, pid) for s in SEEDS}
    cams = {s: c for s, c in cams.items() if c is not None}
    if len(cams) < 2:
        continue
    for a, b in pairs:
        if a not in cams or b not in cams:
            continue
        rho, _ = spearmanr(cams[a].ravel(), cams[b].ravel())
        ya, xa = np.unravel_index(np.argmax(cams[a]), cams[a].shape)
        yb, xb = np.unravel_index(np.argmax(cams[b]), cams[b].shape)
        agr_rows.append({"patientId": pid, "seed_a": a, "seed_b": b,
                         "spearman": float(rho),
                         "peak_px": float(np.hypot(ya - yb, xa - xb) * px_per_cell)})

adf = pd.DataFrame(agr_rows)
adf.to_csv(os.path.join(AGG_DIR, "cam_agreement.csv"), index=False)

wide = preds.pivot_table(index="patientId", columns="seed", values="pred_label")
unanimous = float((wide.nunique(axis=1) == 1).mean())

print(f"[{CKPT_TAG}] inter-seed agreement ({len(pairs)} pairs, at 7x7 native)")
print(f"  spearman  {adf.spearman.mean():.4f} +/- {adf.spearman.std():.4f}")
print(f"  peak_px   {adf.peak_px.mean():.1f} +/- {adf.peak_px.std():.1f} "
      f"(in 1024-px units)")
print(f"  seeds unanimous on the label (thr 0.5): {unanimous:.1%}")
print(f"  mean prob SD across seeds: "
      f"{preds.groupby('patientId').pred_prob.std().mean():.4f}")

#save
summary = {
    "status": "exploratory (best checkpoint; §7 puts this out of scope)",
    "checkpoint": CKPT_TAG, "score_size": SCORE_SIZE,
    "n_seeds": len(SEEDS), "n_images": len(masks),
    "n_all_zero_maps": len(zero_maps),
    "auprc": {"mean_across_seeds": float(per_seed.mean()),
              "spread_sd_of_per_seed_mean": spread,
              "bootstrap_ci_95": [float(lo), float(hi)],
              "per_seed_min": float(per_seed.min()),
              "per_seed_max": float(per_seed.max())},
    "per_image_sd": {"median": float(med), "iqr": [float(q1), float(q3)]},
    "baseline_random": float(sdf.prevalence.mean()),
    "baseline_centre_gaussian": float(np.mean(centre_scores)),
    "pointing_hit": {"mean": float(psdf.pointing_hit.mean()),
                     "std": float(psdf.pointing_hit.std())},
    "agreement": {"spearman": {"mean": float(adf.spearman.mean()),
                               "std": float(adf.spearman.std())},
                  "peak_px": {"mean": float(adf.peak_px.mean()),
                              "std": float(adf.peak_px.std())}},
    "label_unanimity": unanimous,
}
with open(os.path.join(AGG_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].bar(range(len(psdf)), psdf.auprc_mean, color="steelblue")
ax[0].axhline(sdf.prevalence.mean(), color="r", ls="--", label="random baseline")
ax[0].axhline(np.mean(centre_scores), color="orange", ls="--", label="centre baseline")
ax[0].set_xticks(range(len(psdf)))
ax[0].set_xticklabels(psdf.seed, rotation=90, fontsize=7)
ax[0].set_ylabel("mean AUPRC"); ax[0].set_title(f"per-seed AUPRC ({CKPT_TAG})")
ax[0].legend(fontsize=8)

ax[1].hist(per_img["std"], bins=40, color="seagreen")
ax[1].axvline(med, color="r", ls="--", label=f"median {med:.3f}")
ax[1].set_xlabel("per-image SD across seeds")
ax[1].set_title("per-image variance"); ax[1].legend(fontsize=8)

ax[2].bar(range(len(psdf)), psdf.pointing_hit, color="indianred")
ax[2].set_xticks(range(len(psdf)))
ax[2].set_xticklabels(psdf.seed, rotation=90, fontsize=7)
ax[2].set_ylabel("hit rate"); ax[2].set_title(f"pointing game ({CKPT_TAG})")

plt.tight_layout()
plt.savefig(os.path.join(AGG_DIR, "metrics.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nper-seed  -> {OUT_DIR}/seed_*/")
print(f"aggregate -> {AGG_DIR}")


test

In [ ]:
from sklearn.metrics import (precision_score, recall_score, fbeta_score,
                             f1_score, accuracy_score)

THRESHOLD_DIR = os.path.join(XRV_SAVE_DIR, "threshold_sweep_fixedrelu")
os.makedirs(THRESHOLD_DIR, exist_ok=True)

THRESHOLDS = np.round(np.arange(0.20, 0.81, 0.05), 2)
TAGS = ["best", "epoch25"]

rows = []
for tag in TAGS:
    for s in HPARAMS2["seeds"]:
        rn = f"{HPARAMS2['run_name']}_seed{s}"
        f = os.path.join(XRV_SAVE_DIR, f"{rn}_{tag}_test_probs.npz")
        if not os.path.exists(f):
            continue
        d = np.load(f)
        yt, yp = d["y_true"], d["y_prob"]
        for thr in THRESHOLDS:
            pred = (yp >= thr).astype(int)
            rows.append({
                "checkpoint": tag, "seed": s, "thr": float(thr),
                "accuracy":  accuracy_score(yt, pred),
                "precision": precision_score(yt, pred, zero_division=0),
                "recall":    recall_score(yt, pred, zero_division=0),
                "f1":        f1_score(yt, pred, zero_division=0),
                "f2":        fbeta_score(yt, pred, beta=2, zero_division=0),
            })

if not rows:
    print("No *_test_probs.npz files found")
else:
    tdf = pd.DataFrame(rows)
    tdf.to_csv(os.path.join(THRESHOLD_DIR, "per_seed.csv"), index=False)

    for tag in TAGS:
        sub = tdf[tdf.checkpoint == tag]
        if sub.empty:
            continue
        print(f"Threshold Sweep: {tag.capitalize()}  ({sub.seed.nunique()} seeds)")
        agg = (sub.groupby("thr")[["accuracy", "precision", "recall", "f1", "f2"]]
               .agg(["mean", "std"]).round(4))
        print(agg.to_string())

        best_f2 = sub.groupby("thr").f2.mean().idxmax()
        print(f"\n  highest mean F2 at thr {best_f2:.2f} "
              f"(sensitivity only: do not report this as a tuned threshold)")

    agg_out = (tdf.groupby(["checkpoint", "thr"])
               [["accuracy", "precision", "recall", "f1", "f2"]]
               .agg(["mean", "std"]))
    agg_out.to_csv(os.path.join(THRESHOLD_DIR, "threshold_sweep_summary.csv"))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for tag, col in [("best", "steelblue"), ("epoch25", "indianred")]:
        sub = tdf[tdf.checkpoint == tag]
        if sub.empty:
            continue
        g = sub.groupby("thr")[["precision", "recall", "f2"]].mean()
        ax[0].plot(g.index, g.precision, color=col, ls="--",
                   label=f"{tag} precision")
        ax[0].plot(g.index, g.recall, color=col, label=f"{tag} recall")
        ax[1].plot(g.index, g.f2, color=col, label=f"{tag} F2")

    ax[0].axvline(0.5, color="grey", ls=":", lw=1)
    ax[0].set_xlabel("threshold"); ax[0].set_title("precision / recall")
    ax[0].legend(fontsize=8)
    ax[1].axvline(0.5, color="grey", ls=":", lw=1)
    ax[1].set_xlabel("threshold"); ax[1].set_title("F2")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(THRESHOLD_DIR, "threshold_sweep.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nsaved to {THRESHOLD_DIR}")


In [ ]:
import os, json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr, levene
import matplotlib.pyplot as plt

BEST_AGG = os.path.join(XRV_SAVE_DIR, "gradcam_metrics_best_fixedrelu", "_aggregate")
E25_AGG  = os.path.join(XRV_SAVE_DIR, "gradcam_metrics_25_fixedrelu", "_aggregate") # Corrected directory name
OUT_DIR  = os.path.join(XRV_SAVE_DIR, "checkpoint_comparison_fixedrelu")
os.makedirs(OUT_DIR, exist_ok=True)

N_BOOT = 1000
rng = np.random.default_rng(42)

b = pd.read_csv(os.path.join(BEST_AGG, "per_seed_summary.csv"))
e = pd.read_csv(os.path.join(E25_AGG, "per_seed_summary.csv"))

epochs = []
for s in b.seed:
    p = os.path.join(XRV_SAVE_DIR,
                     f"{HPARAMS2['run_name']}_seed{s}_best_metrics.json")
    epochs.append(json.load(open(p)).get("best_epoch") if os.path.exists(p) else np.nan)
b = b.assign(best_epoch=epochs)

m = b.merge(e, on="seed", suffixes=('_best', '_e25'))
m["d_auprc"] = m.auprc_mean_best - m.auprc_mean_e25
m["d_point"] = m.pointing_hit_best - m.pointing_hit_e25
m.to_csv(os.path.join(OUT_DIR, "paired_per_seed.csv"), index=False)

print("Best vs Epoch25, paired over seeds (Exploratory, §7 out of scope)")
print(m[["seed", "best_epoch", "auprc_mean_best", "auprc_mean_e25", "d_auprc",
         "pointing_hit_best", "pointing_hit_e25"]].round(4).to_string(index=False))

print(f"\n  best epoch: median {np.nanmedian(m.best_epoch):.0f}, "
      f"range {np.nanmin(m.best_epoch):.0f}-{np.nanmax(m.best_epoch):.0f}")

print("Level")
for name, cb, ce in [("AUPRC", "auprc_mean_best", "auprc_mean_e25"),
                     ("pointing", "pointing_hit_best", "pointing_hit_e25")]:
    diff = m[cb] - m[ce]
    stat, p = wilcoxon(m[cb], m[ce])
    boot = np.array([rng.choice(diff.values, len(diff), replace=True).mean()
                     for _ in range(N_BOOT)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"  {name:<9s} best {m[cb].mean():.4f}   ep25 {m[ce].mean():.4f}   "
          f"diff {diff.mean():+.4f} [{lo:+.4f}, {hi:+.4f}]   "
          f"Wilcoxon p={p:.4f}")

print("Spread")
for name, cb, ce in [("AUPRC", "auprc_mean_best", "auprc_mean_e25"),
                     ("pointing", "pointing_hit_best", "pointing_hit_e25")]:
    sb, se = m[cb].std(ddof=1), m[ce].std(ddof=1)
    boot = np.array([
        (pd.Series(rng.choice(m[cb].values, len(m), replace=True)).std(ddof=1)
         - pd.Series(rng.choice(m[ce].values, len(m), replace=True)).std(ddof=1))
        for _ in range(N_BOOT)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    _, p = levene(m[cb], m[ce])
    print(f"  {name:<9s} SD best {sb:.4f}   SD ep25 {se:.4f}   "
          f"diff {sb - se:+.4f} [{lo:+.4f}, {hi:+.4f}]   Levene p={p:.4f}")

print("\n  If the best-checkpoint SD is larger, part of that is the selection")
print("  confound above, not extra seed sensitivity.")

rho_a, p_a = spearmanr(m.auprc_mean_best, m.auprc_mean_e25)
rho_p, p_p = spearmanr(m.pointing_hit_best, m.pointing_hit_e25)

print("Seed consistency")
print(f"  AUPRC    Spearman rho {rho_a:+.4f} (p={p_a:.4f})")
print(f"  pointing Spearman rho {rho_p:+.4f} (p={p_p:.4f})")

pb = pd.read_csv(os.path.join(BEST_AGG, "per_image_per_seed_scores.csv"))
pe = pd.read_csv(os.path.join(E25_AGG, "per_image_per_seed_scores.csv"))

ib = pb.groupby("patientId").auprc.agg(["mean", "std"])
ie = pe.groupby("patientId").auprc.agg(["mean", "std"])
joined = ib.join(ie, lsuffix="_best", rsuffix="_e25", how="inner")
joined.to_csv(os.path.join(OUT_DIR, "paired_per_image.csv"))

rho_m, _ = spearmanr(joined["mean_best"], joined["mean_e25"])
rho_s, _ = spearmanr(joined["std_best"], joined["std_e25"])

print(f"Per-image ({len(joined)} images)")
print(f"  image difficulty  correlates rho {rho_m:+.4f}")
print(f"  image instability correlates rho {rho_s:+.4f}")
print(f"  median per-image SD: best {joined['std_best'].median():.4f}   "
      f"ep25 {joined['std_e25'].median():.4f}")

fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))

x = np.arange(len(m))
ax[0].bar(x - 0.2, m.auprc_mean_best, 0.4, label="best", color="steelblue")
ax[0].bar(x + 0.2, m.auprc_mean_e25, 0.4, label="epoch25", color="indianred")
ax[0].set_xticks(x); ax[0].set_xticklabels(m.seed, rotation=90, fontsize=7)
ax[0].set_ylabel("mean AUPRC"); ax[0].set_title("per-seed AUPRC"); ax[0].legend(fontsize=8)

for _, r in m.iterrows():
    ax[1].plot([0, 1], [r.auprc_mean_best, r.auprc_mean_e25],
               color="grey", alpha=0.5, marker="o", ms=3)
ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["best", "epoch25"])
ax[1].set_ylabel("mean AUPRC"); ax[1].set_title("paired shift per seed")

ax[2].scatter(m.auprc_mean_best, m.auprc_mean_e25, s=30, color="seagreen")
lim = [min(m.auprc_mean_best.min(), m.auprc_mean_e25.min()) - 0.01,
       max(m.auprc_mean_best.max(), m.auprc_mean_e25.max()) + 0.01]
ax[2].plot(lim, lim, "k--", lw=1)
ax[2].set_xlabel("best"); ax[2].set_ylabel("epoch25")
ax[2].set_title(f"seed consistency (rho {rho_a:+.2f})")

ax[3].scatter(m.best_epoch, m.auprc_mean_best, s=30, color="darkorange")
ax[3].set_xlabel("epoch the best checkpoint came from")
ax[3].set_ylabel("mean AUPRC (best)")
ax[3].set_title("the selection confound")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "best_vs_epoch25.png"), dpi=150,
            bbox_inches="tight")
plt.show()

summary = {
    "status": "exploratory, §7 puts checkpoint comparison out of scope",
    "n_seeds": len(m),
    "best_epoch": {
        "median": float(np.nanmedian(m.best_epoch)),
        "min": float(np.nanmin(m.best_epoch)),
        "max": float(np.nanmax(m.best_epoch))
    },
    "auprc": {
        "mean_best": float(m.auprc_mean_best.mean()),
        "mean_e25": float(m.auprc_mean_e25.mean()),
        "sd_best": float(m.auprc_mean_best.std(ddof=1)),
        "sd_e25": float(m.auprc_mean_e25.std(ddof=1)),
        "seed_rank_rho": float(rho_a),
    },
    "pointing": {
        "mean_best": float(m.pointing_hit_best.mean()),
        "mean_e25": float(m.pointing_hit_e25.mean()),
        "sd_best": float(m.pointing_hit_best.std(ddof=1)),
        "sd_e25": float(m.pointing_hit_e25.std(ddof=1)),
        "seed_rank_rho": float(rho_p),
    },
    "per_image": {
        "difficulty_rho": float(rho_m),
        "instability_rho": float(rho_s)
    },
}

with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print(f"\nsaved -> {OUT_DIR}")
